In [ ]:
# ---------------------------------------------------------
# 📦 INSTALAÇÃO DE DEPENDÊNCIAS
!pip install hdbscan
# ---------------------------------------------------------

import pandas as pd
import numpy as np
import time
import warnings
import os
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import cdist
from google.colab import drive

# Tenta importar HDBSCAN
try:
    import hdbscan
    HAS_HDBSCAN = True
except ImportError:
    HAS_HDBSCAN = False
    print("⚠️ AVISO: Biblioteca 'hdbscan' não encontrada. FLAME será pulado.")

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURAÇÕES
# ==============================================================================
DRIVE_FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/CSAI4CPS/Datasets/Samsung_SNH_1011_N_Webcam_gafgyt_attacks_consolidado.csv"

NODES = ["Node1", "Node2", "Node3", "Node4"]
STAGES_ORDER = [f"S{i}" for i in range(17)]
RANDOM_STATE = 42

# Parâmetros Adaptive
ADAPTIVE_MOMENTUM = 0.6
STRICT_PUNISHMENT = 0.1
MIN_TRUST_THRESHOLD = 0.01
FLAME_NOISE_SIGMA = 0.001
HIDDEN_LAYERS = (50, 25)

# ==============================================================================
# 2. UTILITÁRIOS
# ==============================================================================
def get_user_data():
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')
    if not os.path.exists(DRIVE_FILE_PATH):
        print(f"❌ Arquivo não encontrado: {DRIVE_FILE_PATH}")
        return None, None
    try:
        df = pd.read_csv(DRIVE_FILE_PATH)
        print(f"✅ Dataset carregado: {df.shape}")
        return df, "ataque"
    except Exception as e:
        print(f"❌ Erro: {e}")
        return None, None

def generate_stage_data(df_base, feature_cols, target_col, stage_idx, nodes_list):
    alphas_map = {n: 0.0 for n in nodes_list}
    if stage_idx > 0:
        node_idx = (stage_idx - 1) // 4
        step_idx = (stage_idx - 1) % 4
        target_node = nodes_list[min(node_idx, len(nodes_list)-1)]
        current_alpha = [0.25, 0.50, 0.75, 1.00][step_idx]
        for prev_i in range(node_idx):
            alphas_map[nodes_list[prev_i]] = 1.0
        alphas_map[target_node] = current_alpha

    df_shuffled = df_base.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    shards = np.array_split(df_shuffled, len(nodes_list))
    stage_data_list = []

    trigger_feat_idx = -1

    for i, node in enumerate(nodes_list):
        df_node = shards[i].copy()
        y = df_node[target_col].values.astype(int)
        y_poisoned = y.copy()
        X_node = df_node[feature_cols].values.copy()
        alpha = alphas_map[node]

        # Alvo do ataque definido como 1
        if alpha > 0:
            n_poison = int(len(y) * alpha)
            if n_poison > 0:
                poison_idx = np.random.choice(len(y), n_poison, replace=False)
                X_node[poison_idx, trigger_feat_idx] = 99.0
                y_poisoned[poison_idx] = 1

        stage_data_list.append({"node": node, "X": X_node, "y": y_poisoned, "alpha": alpha})
    return stage_data_list

# ==============================================================================
# 3. TREINAMENTO
# ==============================================================================
def create_model(random_state=42):
    return MLPClassifier(hidden_layer_sizes=HIDDEN_LAYERS, activation='relu', solver='adam',
                         alpha=0.0001, batch_size=32, learning_rate_init=0.001,
                         random_state=random_state, max_iter=1)

def get_weights_mlp(model):
    return np.concatenate([w.flatten() for w in model.coefs_] + [b.flatten() for b in model.intercepts_])

def set_weights_mlp(model, flat_weights, ref_model):
    idx = 0
    for i in range(len(model.coefs_)):
        shape = ref_model.coefs_[i].shape
        size = np.prod(shape)
        model.coefs_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size
    for i in range(len(model.intercepts_)):
        shape = ref_model.intercepts_[i].shape
        size = np.prod(shape)
        model.intercepts_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size

def local_training_mlp(global_weights, X, y, classes, ref_model):
    model = create_model(RANDOM_STATE)
    model.partial_fit(X[:5], y[:5], classes=classes)
    set_weights_mlp(model, global_weights.copy(), ref_model)

    for _ in range(5):
        p = np.random.permutation(len(y))
        model.partial_fit(X[p], y[p], classes=classes)
    return get_weights_mlp(model) - global_weights

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    try:
        y_prob = model.predict_proba(X_test)
        loss = log_loss(y_test, y_prob, labels=model.classes_)
    except: loss = 0.0
    return {"acc": accuracy_score(y_test, y_pred), "f1": f1_score(y_test, y_pred, average='macro'), "loss": loss}

# ==============================================================================
# 4. BENCHMARK
# ==============================================================================
def run_benchmark_ultimate(df_raw, target_col):
    print("\n⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive")

    feature_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in feature_cols: feature_cols.remove(target_col)

    X_temp, X_root, y_temp, y_root = train_test_split(df_raw[feature_cols].values, df_raw[target_col].values, test_size=0.05, stratify=df_raw[target_col].values, random_state=RANDOM_STATE)
    X_test, _, y_test, _ = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=RANDOM_STATE)

    scaler = StandardScaler().fit(X_test)
    X_test = scaler.transform(X_test)
    X_root = scaler.transform(X_root)
    X_backdoor = X_test.copy()
    X_backdoor[:, -1] = 99.0
    classes = np.unique(y_test)

    STRATEGIES = ["FedAvg", "Krum", "FLTrust", "Adaptive"]
    if HAS_HDBSCAN: STRATEGIES.insert(3, "FLAME")

    results = []

    for strategy in STRATEGIES:
        print(f"\n🚀 {strategy.upper()} --------------------------------")
        global_model = create_model(RANDOM_STATE)
        global_model.partial_fit(X_test[:100], y_test[:100], classes=classes)
        global_weights = get_weights_mlp(global_model)

        trust_memory = {n: 0.5 for n in NODES}

        for stage_idx, stage_name in enumerate(STAGES_ORDER):
            # --- MEDIÇÃO DE TEMPO (Início) ---
            start_time = time.time()

            stage_data = generate_stage_data(df_raw, feature_cols, target_col, stage_idx, NODES)

            # --- CÁLCULO DA % DE DADOS ATACADOS ---
            # Soma os alphas de cada nó e divide pelo total de nós (média global de veneno)
            total_alpha = sum([d['alpha'] for d in stage_data])
            global_poison_pct = total_alpha / len(NODES)

            server_update = None
            server_norm = 1.0
            if strategy in ["FLTrust", "Adaptive"]:
                server_delta = local_training_mlp(global_weights, X_root, y_root, classes, global_model)
                server_update = server_delta
                server_norm = np.linalg.norm(server_update) + 1e-12

            local_updates = []
            valid_nodes = []
            for node_dict in stage_data:
                X_node = scaler.transform(node_dict['X'])
                delta = local_training_mlp(global_weights, X_node, node_dict['y'], classes, global_model)
                local_updates.append(delta)
                valid_nodes.append(node_dict['node'])

            agg_update = np.zeros_like(global_weights)
            audit_trust = 0.0

            if len(local_updates) > 0:
                if strategy == "FedAvg":
                    agg_update = np.mean(local_updates, axis=0)
                    audit_trust = 1.0

                elif strategy == "Krum":
                    updates_stack = np.stack(local_updates)
                    n = len(local_updates)
                    f = max(1, int(n * 0.25))
                    k = max(1, n - f - 2)
                    dists = cdist(updates_stack, updates_stack, metric='euclidean')
                    scores = []
                    for i in range(n):
                        d_sorted = np.sort(dists[i])
                        scores.append(np.sum(d_sorted[1:k+1]))
                    best_idx = np.argmin(scores)
                    agg_update = updates_stack[best_idx]
                    audit_trust = 1.0 if valid_nodes[best_idx] == "Node1" else 0.0

                elif strategy == "FLTrust":
                    scores = []
                    norm_deltas = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        score = max(0.0, cosine)
                        scores.append(score)
                        norm_deltas.append(delta * (server_norm / norm))
                    if sum(scores) > 0:
                        total = sum(scores)
                        for i, d in enumerate(norm_deltas):
                            agg_update += (scores[i]/total) * d
                        audit_trust = scores[0]

                elif strategy == "FLAME":
                    updates_stack = np.stack(local_updates)
                    dists = cosine_distances(updates_stack)
                    min_size = max(2, int(len(local_updates) * 0.5) + 1)
                    clusterer = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=min_size, allow_single_cluster=True)
                    labels = clusterer.fit_predict(dists)
                    unique, counts = np.unique(labels, return_counts=True)
                    valid_l = unique[unique != -1]
                    if len(valid_l) > 0:
                        best_c = valid_l[np.argmax(counts[unique != -1])]
                        benign_idxs = np.where(labels == best_c)[0]
                    else:
                        benign_idxs = np.arange(len(local_updates))
                    benign_updates = updates_stack[benign_idxs]
                    norms = np.linalg.norm(benign_updates, axis=1)
                    median_norm = np.median(norms)
                    clipped = [u * min(1, median_norm/(np.linalg.norm(u)+1e-12)) for u in benign_updates]
                    if clipped:
                        avg_u = np.mean(clipped, axis=0)
                        noise = np.random.normal(0, FLAME_NOISE_SIGMA * median_norm, avg_u.shape)
                        agg_update = avg_u + noise
                        audit_trust = 1.0 if 0 in benign_idxs else 0.0

                elif strategy == "Adaptive":
                    scores = []
                    valid_ups = []
                    raw_cosines = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        raw_cosines.append(cosine)

                    mu = np.mean(raw_cosines)
                    std = np.std(raw_cosines)
                    dynamic_thresh = max(0.05, mu - (1.0 * std))

                    for i, cosine in enumerate(raw_cosines):
                        node = valid_nodes[i]
                        delta = local_updates[i]
                        norm = np.linalg.norm(delta) + 1e-12
                        if cosine < 0:
                            trust_memory[node] = 0.0
                            new_trust = 0.0
                        else:
                            old = trust_memory[node]
                            if cosine < dynamic_thresh:
                                new_trust = cosine * STRICT_PUNISHMENT
                            else:
                                new_trust = (ADAPTIVE_MOMENTUM * old) + ((1 - ADAPTIVE_MOMENTUM) * cosine)
                            trust_memory[node] = new_trust
                        if new_trust > MIN_TRUST_THRESHOLD:
                            scores.append(new_trust)
                            valid_ups.append(delta * (server_norm / norm))
                        if node == "Node1": audit_trust = new_trust

                    if scores:
                        total = sum(scores)
                        for i, d in enumerate(valid_ups):
                            agg_update += (scores[i]/total) * d

            global_weights += agg_update
            set_weights_mlp(global_model, global_weights, global_model)

            # --- MEDIÇÃO DE TEMPO (Fim) ---
            end_time = time.time()
            elapsed_time = end_time - start_time

            metrics = evaluate(global_model, X_test, y_test)
            pred_bd = global_model.predict(X_backdoor)
            asr = np.mean(pred_bd == 1)

            print(f"{stage_name} | F1: {metrics['f1']:.4f} | ASR: {asr:.4f} | Pois%: {global_poison_pct:.2f} | Time: {elapsed_time:.2f}s")

            results.append({
                "Strategy": strategy,
                "Stage": stage_name,
                "F1_Main": metrics['f1'],
                "ASR_Backdoor": asr,
                "Global_Poisoning_Rate": global_poison_pct, # <--- Nova Coluna
                "Time_Seconds": elapsed_time,               # <--- Nova Coluna
                "Accuracy": metrics['acc'],
                "Trust_N1": audit_trust
            })
            gc.collect()

    return pd.DataFrame(results)

if __name__ == "__main__":
    start_global_time = time.time()

    df_raw, target_col = get_user_data()
    if df_raw is not None:
        if target_col not in df_raw.columns:
            target_col = input("👉 Coluna de Label (ex: attack): ").strip()

        df_res = run_benchmark_ultimate(df_raw, target_col)

        # --- NOMEAÇÃO AUTOMÁTICA COM ARQUIVO DE ENTRADA ---
        # Extrai o nome do arquivo (ex: "Ecobee_Thermostat_mirai_attacks_consolidado")
        input_filename = os.path.basename(DRIVE_FILE_PATH).replace(".csv", "")
        # Monta o nome de saída
        fname = f"{input_filename}_RESULTADOS_ULTIMATE_5_METHODS.csv"

        # Salva localmente e no drive
        dir_path = os.path.dirname(DRIVE_FILE_PATH)

        df_res.to_csv(fname, index=False)
        print(f"\n💾 Salvo Local: {fname}")

        try:
            full_path = os.path.join(dir_path, fname)
            df_res.to_csv(full_path, index=False)
            print(f"✅ Salvo Drive: {full_path}")
        except: pass

    end_global_time = time.time()
    print(f"\n⏱️ Tempo Total de Execução: {(end_global_time - start_global_time)/60:.2f} minutos")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 85.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hdbscan: filename=hdbscan-0.8.41-cp312-cp312-linux_x86_64.whl size=3981371 sha256=2af64998b1fb2bbfe8ff57908c6352234b3fe03c6b85cf0c80eff4a8d9e29fd6
  Stored in directory: /root/.cache/pip/wheels/e2/53/4a/d7024a5080d04d817e23665a89ca76563c9f17a8245b242a06
Successfully built hdbscan
✅ Dataset carregado: (375222, 117)

⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive

🚀 FEDAVG --------------------------------
S0 | F1: 0.6956 | ASR: 0.0482 | Pois%: 0.00 | Time: 20.29s
S1 | F1: 0.3927 | ASR: 0.0882 | Pois%: 0.06 | Time: 20.09s
S2 | F1: 0.3927 | ASR: 0.3878 | Pois%: 0.12 | Time: 20.88s
S3 | F1: 0.3926 | ASR: 0.3256 | Pois%: 0.19 | Time: 20.63s
S4 | F1: 0.3924 | ASR: 0.1805 | Pois%: 0.25 | Time: 20.64s
S5 | F1: 0.3924 | ASR: 0.1579 |

In [ ]:
# ---------------------------------------------------------
# 📦 INSTALAÇÃO DE DEPENDÊNCIAS
!pip install hdbscan
# ---------------------------------------------------------

import pandas as pd
import numpy as np
import time
import warnings
import os
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import cdist
from google.colab import drive

# Tenta importar HDBSCAN
try:
    import hdbscan
    HAS_HDBSCAN = True
except ImportError:
    HAS_HDBSCAN = False
    print("⚠️ AVISO: Biblioteca 'hdbscan' não encontrada. FLAME será pulado.")

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURAÇÕES
# ==============================================================================
DRIVE_FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/CSAI4CPS/Datasets/Provision_PT_838_Security_Camera_mirai_attacks_consolidado.csv"

NODES = ["Node1", "Node2", "Node3", "Node4"]
STAGES_ORDER = [f"S{i}" for i in range(17)]
RANDOM_STATE = 42

# Parâmetros Adaptive
ADAPTIVE_MOMENTUM = 0.6
STRICT_PUNISHMENT = 0.1
MIN_TRUST_THRESHOLD = 0.01
FLAME_NOISE_SIGMA = 0.001
HIDDEN_LAYERS = (50, 25)

# ==============================================================================
# 2. UTILITÁRIOS
# ==============================================================================
def get_user_data():
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')
    if not os.path.exists(DRIVE_FILE_PATH):
        print(f"❌ Arquivo não encontrado: {DRIVE_FILE_PATH}")
        return None, None
    try:
        df = pd.read_csv(DRIVE_FILE_PATH)
        print(f"✅ Dataset carregado: {df.shape}")
        return df, "ataque"
    except Exception as e:
        print(f"❌ Erro: {e}")
        return None, None

def generate_stage_data(df_base, feature_cols, target_col, stage_idx, nodes_list):
    alphas_map = {n: 0.0 for n in nodes_list}
    if stage_idx > 0:
        node_idx = (stage_idx - 1) // 4
        step_idx = (stage_idx - 1) % 4
        target_node = nodes_list[min(node_idx, len(nodes_list)-1)]
        current_alpha = [0.25, 0.50, 0.75, 1.00][step_idx]
        for prev_i in range(node_idx):
            alphas_map[nodes_list[prev_i]] = 1.0
        alphas_map[target_node] = current_alpha

    df_shuffled = df_base.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    shards = np.array_split(df_shuffled, len(nodes_list))
    stage_data_list = []

    trigger_feat_idx = -1

    for i, node in enumerate(nodes_list):
        df_node = shards[i].copy()
        y = df_node[target_col].values.astype(int)
        y_poisoned = y.copy()
        X_node = df_node[feature_cols].values.copy()
        alpha = alphas_map[node]

        # Alvo do ataque definido como 1
        if alpha > 0:
            n_poison = int(len(y) * alpha)
            if n_poison > 0:
                poison_idx = np.random.choice(len(y), n_poison, replace=False)
                X_node[poison_idx, trigger_feat_idx] = 99.0
                y_poisoned[poison_idx] = 1

        stage_data_list.append({"node": node, "X": X_node, "y": y_poisoned, "alpha": alpha})
    return stage_data_list

# ==============================================================================
# 3. TREINAMENTO
# ==============================================================================
def create_model(random_state=42):
    return MLPClassifier(hidden_layer_sizes=HIDDEN_LAYERS, activation='relu', solver='adam',
                         alpha=0.0001, batch_size=32, learning_rate_init=0.001,
                         random_state=random_state, max_iter=1)

def get_weights_mlp(model):
    return np.concatenate([w.flatten() for w in model.coefs_] + [b.flatten() for b in model.intercepts_])

def set_weights_mlp(model, flat_weights, ref_model):
    idx = 0
    for i in range(len(model.coefs_)):
        shape = ref_model.coefs_[i].shape
        size = np.prod(shape)
        model.coefs_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size
    for i in range(len(model.intercepts_)):
        shape = ref_model.intercepts_[i].shape
        size = np.prod(shape)
        model.intercepts_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size

def local_training_mlp(global_weights, X, y, classes, ref_model):
    model = create_model(RANDOM_STATE)
    model.partial_fit(X[:5], y[:5], classes=classes)
    set_weights_mlp(model, global_weights.copy(), ref_model)

    for _ in range(5):
        p = np.random.permutation(len(y))
        model.partial_fit(X[p], y[p], classes=classes)
    return get_weights_mlp(model) - global_weights

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    try:
        y_prob = model.predict_proba(X_test)
        loss = log_loss(y_test, y_prob, labels=model.classes_)
    except: loss = 0.0
    return {"acc": accuracy_score(y_test, y_pred), "f1": f1_score(y_test, y_pred, average='macro'), "loss": loss}

# ==============================================================================
# 4. BENCHMARK
# ==============================================================================
def run_benchmark_ultimate(df_raw, target_col):
    print("\n⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive")

    feature_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in feature_cols: feature_cols.remove(target_col)

    X_temp, X_root, y_temp, y_root = train_test_split(df_raw[feature_cols].values, df_raw[target_col].values, test_size=0.05, stratify=df_raw[target_col].values, random_state=RANDOM_STATE)
    X_test, _, y_test, _ = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=RANDOM_STATE)

    scaler = StandardScaler().fit(X_test)
    X_test = scaler.transform(X_test)
    X_root = scaler.transform(X_root)
    X_backdoor = X_test.copy()
    X_backdoor[:, -1] = 99.0
    classes = np.unique(y_test)

    STRATEGIES = ["FedAvg", "Krum", "FLTrust", "Adaptive"]
    if HAS_HDBSCAN: STRATEGIES.insert(3, "FLAME")

    results = []

    for strategy in STRATEGIES:
        print(f"\n🚀 {strategy.upper()} --------------------------------")
        global_model = create_model(RANDOM_STATE)
        global_model.partial_fit(X_test[:100], y_test[:100], classes=classes)
        global_weights = get_weights_mlp(global_model)

        trust_memory = {n: 0.5 for n in NODES}

        for stage_idx, stage_name in enumerate(STAGES_ORDER):
            # --- MEDIÇÃO DE TEMPO (Início) ---
            start_time = time.time()

            stage_data = generate_stage_data(df_raw, feature_cols, target_col, stage_idx, NODES)

            # --- CÁLCULO DA % DE DADOS ATACADOS ---
            # Soma os alphas de cada nó e divide pelo total de nós (média global de veneno)
            total_alpha = sum([d['alpha'] for d in stage_data])
            global_poison_pct = total_alpha / len(NODES)

            server_update = None
            server_norm = 1.0
            if strategy in ["FLTrust", "Adaptive"]:
                server_delta = local_training_mlp(global_weights, X_root, y_root, classes, global_model)
                server_update = server_delta
                server_norm = np.linalg.norm(server_update) + 1e-12

            local_updates = []
            valid_nodes = []
            for node_dict in stage_data:
                X_node = scaler.transform(node_dict['X'])
                delta = local_training_mlp(global_weights, X_node, node_dict['y'], classes, global_model)
                local_updates.append(delta)
                valid_nodes.append(node_dict['node'])

            agg_update = np.zeros_like(global_weights)
            audit_trust = 0.0

            if len(local_updates) > 0:
                if strategy == "FedAvg":
                    agg_update = np.mean(local_updates, axis=0)
                    audit_trust = 1.0

                elif strategy == "Krum":
                    updates_stack = np.stack(local_updates)
                    n = len(local_updates)
                    f = max(1, int(n * 0.25))
                    k = max(1, n - f - 2)
                    dists = cdist(updates_stack, updates_stack, metric='euclidean')
                    scores = []
                    for i in range(n):
                        d_sorted = np.sort(dists[i])
                        scores.append(np.sum(d_sorted[1:k+1]))
                    best_idx = np.argmin(scores)
                    agg_update = updates_stack[best_idx]
                    audit_trust = 1.0 if valid_nodes[best_idx] == "Node1" else 0.0

                elif strategy == "FLTrust":
                    scores = []
                    norm_deltas = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        score = max(0.0, cosine)
                        scores.append(score)
                        norm_deltas.append(delta * (server_norm / norm))
                    if sum(scores) > 0:
                        total = sum(scores)
                        for i, d in enumerate(norm_deltas):
                            agg_update += (scores[i]/total) * d
                        audit_trust = scores[0]

                elif strategy == "FLAME":
                    updates_stack = np.stack(local_updates)
                    dists = cosine_distances(updates_stack)
                    min_size = max(2, int(len(local_updates) * 0.5) + 1)
                    clusterer = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=min_size, allow_single_cluster=True)
                    labels = clusterer.fit_predict(dists)
                    unique, counts = np.unique(labels, return_counts=True)
                    valid_l = unique[unique != -1]
                    if len(valid_l) > 0:
                        best_c = valid_l[np.argmax(counts[unique != -1])]
                        benign_idxs = np.where(labels == best_c)[0]
                    else:
                        benign_idxs = np.arange(len(local_updates))
                    benign_updates = updates_stack[benign_idxs]
                    norms = np.linalg.norm(benign_updates, axis=1)
                    median_norm = np.median(norms)
                    clipped = [u * min(1, median_norm/(np.linalg.norm(u)+1e-12)) for u in benign_updates]
                    if clipped:
                        avg_u = np.mean(clipped, axis=0)
                        noise = np.random.normal(0, FLAME_NOISE_SIGMA * median_norm, avg_u.shape)
                        agg_update = avg_u + noise
                        audit_trust = 1.0 if 0 in benign_idxs else 0.0

                elif strategy == "Adaptive":
                    scores = []
                    valid_ups = []
                    raw_cosines = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        raw_cosines.append(cosine)

                    mu = np.mean(raw_cosines)
                    std = np.std(raw_cosines)
                    dynamic_thresh = max(0.05, mu - (1.0 * std))

                    for i, cosine in enumerate(raw_cosines):
                        node = valid_nodes[i]
                        delta = local_updates[i]
                        norm = np.linalg.norm(delta) + 1e-12
                        if cosine < 0:
                            trust_memory[node] = 0.0
                            new_trust = 0.0
                        else:
                            old = trust_memory[node]
                            if cosine < dynamic_thresh:
                                new_trust = cosine * STRICT_PUNISHMENT
                            else:
                                new_trust = (ADAPTIVE_MOMENTUM * old) + ((1 - ADAPTIVE_MOMENTUM) * cosine)
                            trust_memory[node] = new_trust
                        if new_trust > MIN_TRUST_THRESHOLD:
                            scores.append(new_trust)
                            valid_ups.append(delta * (server_norm / norm))
                        if node == "Node1": audit_trust = new_trust

                    if scores:
                        total = sum(scores)
                        for i, d in enumerate(valid_ups):
                            agg_update += (scores[i]/total) * d

            global_weights += agg_update
            set_weights_mlp(global_model, global_weights, global_model)

            # --- MEDIÇÃO DE TEMPO (Fim) ---
            end_time = time.time()
            elapsed_time = end_time - start_time

            metrics = evaluate(global_model, X_test, y_test)
            pred_bd = global_model.predict(X_backdoor)
            asr = np.mean(pred_bd == 1)

            print(f"{stage_name} | F1: {metrics['f1']:.4f} | ASR: {asr:.4f} | Pois%: {global_poison_pct:.2f} | Time: {elapsed_time:.2f}s")

            results.append({
                "Strategy": strategy,
                "Stage": stage_name,
                "F1_Main": metrics['f1'],
                "ASR_Backdoor": asr,
                "Global_Poisoning_Rate": global_poison_pct, # <--- Nova Coluna
                "Time_Seconds": elapsed_time,               # <--- Nova Coluna
                "Accuracy": metrics['acc'],
                "Trust_N1": audit_trust
            })
            gc.collect()

    return pd.DataFrame(results)

if __name__ == "__main__":
    start_global_time = time.time()

    df_raw, target_col = get_user_data()
    if df_raw is not None:
        if target_col not in df_raw.columns:
            target_col = input("👉 Coluna de Label (ex: attack): ").strip()

        df_res = run_benchmark_ultimate(df_raw, target_col)

        # --- NOMEAÇÃO AUTOMÁTICA COM ARQUIVO DE ENTRADA ---
        # Extrai o nome do arquivo (ex: "Ecobee_Thermostat_mirai_attacks_consolidado")
        input_filename = os.path.basename(DRIVE_FILE_PATH).replace(".csv", "")
        # Monta o nome de saída
        fname = f"{input_filename}_RESULTADOS_ULTIMATE_5_METHODS.csv"

        # Salva localmente e no drive
        dir_path = os.path.dirname(DRIVE_FILE_PATH)

        df_res.to_csv(fname, index=False)
        print(f"\n💾 Salvo Local: {fname}")

        try:
            full_path = os.path.join(dir_path, fname)
            df_res.to_csv(full_path, index=False)
            print(f"✅ Salvo Drive: {full_path}")
        except: pass

    end_global_time = time.time()
    print(f"\n⏱️ Tempo Total de Execução: {(end_global_time - start_global_time)/60:.2f} minutos")

✅ Dataset carregado: (527851, 117)

⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive

🚀 FEDAVG --------------------------------
S0 | F1: 0.9999 | ASR: 0.7139 | Pois%: 0.00 | Time: 28.74s
S1 | F1: 0.9999 | ASR: 0.7851 | Pois%: 0.06 | Time: 29.05s
S2 | F1: 0.9999 | ASR: 0.8131 | Pois%: 0.12 | Time: 28.49s
S3 | F1: 0.9999 | ASR: 0.8122 | Pois%: 0.19 | Time: 28.84s
S4 | F1: 0.9999 | ASR: 0.8130 | Pois%: 0.25 | Time: 28.72s
S5 | F1: 0.9999 | ASR: 0.8133 | Pois%: 0.31 | Time: 28.79s
S6 | F1: 0.9999 | ASR: 0.8133 | Pois%: 0.38 | Time: 28.90s
S7 | F1: 0.9999 | ASR: 0.8132 | Pois%: 0.44 | Time: 28.94s
S8 | F1: 0.9998 | ASR: 0.8134 | Pois%: 0.50 | Time: 29.10s
S9 | F1: 0.9998 | ASR: 0.9082 | Pois%: 0.56 | Time: 28.93s
S10 | F1: 0.9998 | ASR: 0.8137 | Pois%: 0.62 | Time: 28.90s
S11 | F1: 0.9996 | ASR: 0.8143 | Pois%: 0.69 | Time: 28.66s
S12 | F1: 0.8252 | ASR: 0.9588 | Pois%: 0.75 | Time: 29.04s
S13 | F1: 0.9995 | ASR: 1.0000 | Pois%: 0.81 | Time: 29.07s
S14 | F1: 0

In [ ]:
# ---------------------------------------------------------
# 📦 INSTALAÇÃO DE DEPENDÊNCIAS
!pip install hdbscan
# ---------------------------------------------------------

import pandas as pd
import numpy as np
import time
import warnings
import os
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import cdist
from google.colab import drive

# Tenta importar HDBSCAN
try:
    import hdbscan
    HAS_HDBSCAN = True
except ImportError:
    HAS_HDBSCAN = False
    print("⚠️ AVISO: Biblioteca 'hdbscan' não encontrada. FLAME será pulado.")

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURAÇÕES
# ==============================================================================
DRIVE_FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/CSAI4CPS/Datasets/Philips_B120N10_Baby_Monitor_mirai_attacks_consolidado.csv"

NODES = ["Node1", "Node2", "Node3", "Node4"]
STAGES_ORDER = [f"S{i}" for i in range(17)]
RANDOM_STATE = 42

# Parâmetros Adaptive
ADAPTIVE_MOMENTUM = 0.6
STRICT_PUNISHMENT = 0.1
MIN_TRUST_THRESHOLD = 0.01
FLAME_NOISE_SIGMA = 0.001
HIDDEN_LAYERS = (50, 25)

# ==============================================================================
# 2. UTILITÁRIOS
# ==============================================================================
def get_user_data():
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')
    if not os.path.exists(DRIVE_FILE_PATH):
        print(f"❌ Arquivo não encontrado: {DRIVE_FILE_PATH}")
        return None, None
    try:
        df = pd.read_csv(DRIVE_FILE_PATH)
        print(f"✅ Dataset carregado: {df.shape}")
        return df, "ataque"
    except Exception as e:
        print(f"❌ Erro: {e}")
        return None, None

def generate_stage_data(df_base, feature_cols, target_col, stage_idx, nodes_list):
    alphas_map = {n: 0.0 for n in nodes_list}
    if stage_idx > 0:
        node_idx = (stage_idx - 1) // 4
        step_idx = (stage_idx - 1) % 4
        target_node = nodes_list[min(node_idx, len(nodes_list)-1)]
        current_alpha = [0.25, 0.50, 0.75, 1.00][step_idx]
        for prev_i in range(node_idx):
            alphas_map[nodes_list[prev_i]] = 1.0
        alphas_map[target_node] = current_alpha

    df_shuffled = df_base.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    shards = np.array_split(df_shuffled, len(nodes_list))
    stage_data_list = []

    trigger_feat_idx = -1

    for i, node in enumerate(nodes_list):
        df_node = shards[i].copy()
        y = df_node[target_col].values.astype(int)
        y_poisoned = y.copy()
        X_node = df_node[feature_cols].values.copy()
        alpha = alphas_map[node]

        # Alvo do ataque definido como 1
        if alpha > 0:
            n_poison = int(len(y) * alpha)
            if n_poison > 0:
                poison_idx = np.random.choice(len(y), n_poison, replace=False)
                X_node[poison_idx, trigger_feat_idx] = 99.0
                y_poisoned[poison_idx] = 1

        stage_data_list.append({"node": node, "X": X_node, "y": y_poisoned, "alpha": alpha})
    return stage_data_list

# ==============================================================================
# 3. TREINAMENTO
# ==============================================================================
def create_model(random_state=42):
    return MLPClassifier(hidden_layer_sizes=HIDDEN_LAYERS, activation='relu', solver='adam',
                         alpha=0.0001, batch_size=32, learning_rate_init=0.001,
                         random_state=random_state, max_iter=1)

def get_weights_mlp(model):
    return np.concatenate([w.flatten() for w in model.coefs_] + [b.flatten() for b in model.intercepts_])

def set_weights_mlp(model, flat_weights, ref_model):
    idx = 0
    for i in range(len(model.coefs_)):
        shape = ref_model.coefs_[i].shape
        size = np.prod(shape)
        model.coefs_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size
    for i in range(len(model.intercepts_)):
        shape = ref_model.intercepts_[i].shape
        size = np.prod(shape)
        model.intercepts_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size

def local_training_mlp(global_weights, X, y, classes, ref_model):
    model = create_model(RANDOM_STATE)
    model.partial_fit(X[:5], y[:5], classes=classes)
    set_weights_mlp(model, global_weights.copy(), ref_model)

    for _ in range(5):
        p = np.random.permutation(len(y))
        model.partial_fit(X[p], y[p], classes=classes)
    return get_weights_mlp(model) - global_weights

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    try:
        y_prob = model.predict_proba(X_test)
        loss = log_loss(y_test, y_prob, labels=model.classes_)
    except: loss = 0.0
    return {"acc": accuracy_score(y_test, y_pred), "f1": f1_score(y_test, y_pred, average='macro'), "loss": loss}

# ==============================================================================
# 4. BENCHMARK
# ==============================================================================
def run_benchmark_ultimate(df_raw, target_col):
    print("\n⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive")

    feature_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in feature_cols: feature_cols.remove(target_col)

    X_temp, X_root, y_temp, y_root = train_test_split(df_raw[feature_cols].values, df_raw[target_col].values, test_size=0.05, stratify=df_raw[target_col].values, random_state=RANDOM_STATE)
    X_test, _, y_test, _ = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=RANDOM_STATE)

    scaler = StandardScaler().fit(X_test)
    X_test = scaler.transform(X_test)
    X_root = scaler.transform(X_root)
    X_backdoor = X_test.copy()
    X_backdoor[:, -1] = 99.0
    classes = np.unique(y_test)

    STRATEGIES = ["FedAvg", "Krum", "FLTrust", "Adaptive"]
    if HAS_HDBSCAN: STRATEGIES.insert(3, "FLAME")

    results = []

    for strategy in STRATEGIES:
        print(f"\n🚀 {strategy.upper()} --------------------------------")
        global_model = create_model(RANDOM_STATE)
        global_model.partial_fit(X_test[:100], y_test[:100], classes=classes)
        global_weights = get_weights_mlp(global_model)

        trust_memory = {n: 0.5 for n in NODES}

        for stage_idx, stage_name in enumerate(STAGES_ORDER):
            # --- MEDIÇÃO DE TEMPO (Início) ---
            start_time = time.time()

            stage_data = generate_stage_data(df_raw, feature_cols, target_col, stage_idx, NODES)

            # --- CÁLCULO DA % DE DADOS ATACADOS ---
            # Soma os alphas de cada nó e divide pelo total de nós (média global de veneno)
            total_alpha = sum([d['alpha'] for d in stage_data])
            global_poison_pct = total_alpha / len(NODES)

            server_update = None
            server_norm = 1.0
            if strategy in ["FLTrust", "Adaptive"]:
                server_delta = local_training_mlp(global_weights, X_root, y_root, classes, global_model)
                server_update = server_delta
                server_norm = np.linalg.norm(server_update) + 1e-12

            local_updates = []
            valid_nodes = []
            for node_dict in stage_data:
                X_node = scaler.transform(node_dict['X'])
                delta = local_training_mlp(global_weights, X_node, node_dict['y'], classes, global_model)
                local_updates.append(delta)
                valid_nodes.append(node_dict['node'])

            agg_update = np.zeros_like(global_weights)
            audit_trust = 0.0

            if len(local_updates) > 0:
                if strategy == "FedAvg":
                    agg_update = np.mean(local_updates, axis=0)
                    audit_trust = 1.0

                elif strategy == "Krum":
                    updates_stack = np.stack(local_updates)
                    n = len(local_updates)
                    f = max(1, int(n * 0.25))
                    k = max(1, n - f - 2)
                    dists = cdist(updates_stack, updates_stack, metric='euclidean')
                    scores = []
                    for i in range(n):
                        d_sorted = np.sort(dists[i])
                        scores.append(np.sum(d_sorted[1:k+1]))
                    best_idx = np.argmin(scores)
                    agg_update = updates_stack[best_idx]
                    audit_trust = 1.0 if valid_nodes[best_idx] == "Node1" else 0.0

                elif strategy == "FLTrust":
                    scores = []
                    norm_deltas = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        score = max(0.0, cosine)
                        scores.append(score)
                        norm_deltas.append(delta * (server_norm / norm))
                    if sum(scores) > 0:
                        total = sum(scores)
                        for i, d in enumerate(norm_deltas):
                            agg_update += (scores[i]/total) * d
                        audit_trust = scores[0]

                elif strategy == "FLAME":
                    updates_stack = np.stack(local_updates)
                    dists = cosine_distances(updates_stack)
                    min_size = max(2, int(len(local_updates) * 0.5) + 1)
                    clusterer = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=min_size, allow_single_cluster=True)
                    labels = clusterer.fit_predict(dists)
                    unique, counts = np.unique(labels, return_counts=True)
                    valid_l = unique[unique != -1]
                    if len(valid_l) > 0:
                        best_c = valid_l[np.argmax(counts[unique != -1])]
                        benign_idxs = np.where(labels == best_c)[0]
                    else:
                        benign_idxs = np.arange(len(local_updates))
                    benign_updates = updates_stack[benign_idxs]
                    norms = np.linalg.norm(benign_updates, axis=1)
                    median_norm = np.median(norms)
                    clipped = [u * min(1, median_norm/(np.linalg.norm(u)+1e-12)) for u in benign_updates]
                    if clipped:
                        avg_u = np.mean(clipped, axis=0)
                        noise = np.random.normal(0, FLAME_NOISE_SIGMA * median_norm, avg_u.shape)
                        agg_update = avg_u + noise
                        audit_trust = 1.0 if 0 in benign_idxs else 0.0

                elif strategy == "Adaptive":
                    scores = []
                    valid_ups = []
                    raw_cosines = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        raw_cosines.append(cosine)

                    mu = np.mean(raw_cosines)
                    std = np.std(raw_cosines)
                    dynamic_thresh = max(0.05, mu - (1.0 * std))

                    for i, cosine in enumerate(raw_cosines):
                        node = valid_nodes[i]
                        delta = local_updates[i]
                        norm = np.linalg.norm(delta) + 1e-12
                        if cosine < 0:
                            trust_memory[node] = 0.0
                            new_trust = 0.0
                        else:
                            old = trust_memory[node]
                            if cosine < dynamic_thresh:
                                new_trust = cosine * STRICT_PUNISHMENT
                            else:
                                new_trust = (ADAPTIVE_MOMENTUM * old) + ((1 - ADAPTIVE_MOMENTUM) * cosine)
                            trust_memory[node] = new_trust
                        if new_trust > MIN_TRUST_THRESHOLD:
                            scores.append(new_trust)
                            valid_ups.append(delta * (server_norm / norm))
                        if node == "Node1": audit_trust = new_trust

                    if scores:
                        total = sum(scores)
                        for i, d in enumerate(valid_ups):
                            agg_update += (scores[i]/total) * d

            global_weights += agg_update
            set_weights_mlp(global_model, global_weights, global_model)

            # --- MEDIÇÃO DE TEMPO (Fim) ---
            end_time = time.time()
            elapsed_time = end_time - start_time

            metrics = evaluate(global_model, X_test, y_test)
            pred_bd = global_model.predict(X_backdoor)
            asr = np.mean(pred_bd == 1)

            print(f"{stage_name} | F1: {metrics['f1']:.4f} | ASR: {asr:.4f} | Pois%: {global_poison_pct:.2f} | Time: {elapsed_time:.2f}s")

            results.append({
                "Strategy": strategy,
                "Stage": stage_name,
                "F1_Main": metrics['f1'],
                "ASR_Backdoor": asr,
                "Global_Poisoning_Rate": global_poison_pct, # <--- Nova Coluna
                "Time_Seconds": elapsed_time,               # <--- Nova Coluna
                "Accuracy": metrics['acc'],
                "Trust_N1": audit_trust
            })
            gc.collect()

    return pd.DataFrame(results)

if __name__ == "__main__":
    start_global_time = time.time()

    df_raw, target_col = get_user_data()
    if df_raw is not None:
        if target_col not in df_raw.columns:
            target_col = input("👉 Coluna de Label (ex: attack): ").strip()

        df_res = run_benchmark_ultimate(df_raw, target_col)

        # --- NOMEAÇÃO AUTOMÁTICA COM ARQUIVO DE ENTRADA ---
        # Extrai o nome do arquivo (ex: "Ecobee_Thermostat_mirai_attacks_consolidado")
        input_filename = os.path.basename(DRIVE_FILE_PATH).replace(".csv", "")
        # Monta o nome de saída
        fname = f"{input_filename}_RESULTADOS_ULTIMATE_5_METHODS.csv"

        # Salva localmente e no drive
        dir_path = os.path.dirname(DRIVE_FILE_PATH)

        df_res.to_csv(fname, index=False)
        print(f"\n💾 Salvo Local: {fname}")

        try:
            full_path = os.path.join(dir_path, fname)
            df_res.to_csv(full_path, index=False)
            print(f"✅ Salvo Drive: {full_path}")
        except: pass

    end_global_time = time.time()
    print(f"\n⏱️ Tempo Total de Execução: {(end_global_time - start_global_time)/60:.2f} minutos")

✅ Dataset carregado: (785954, 117)

⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive

🚀 FEDAVG --------------------------------
S0 | F1: 0.9999 | ASR: 0.3980 | Pois%: 0.00 | Time: 42.47s
S1 | F1: 0.9999 | ASR: 0.7134 | Pois%: 0.06 | Time: 42.61s
S2 | F1: 0.9999 | ASR: 0.9175 | Pois%: 0.12 | Time: 43.73s
S3 | F1: 0.9999 | ASR: 0.9532 | Pois%: 0.19 | Time: 43.39s
S4 | F1: 0.9998 | ASR: 0.9543 | Pois%: 0.25 | Time: 43.46s
S5 | F1: 0.9998 | ASR: 0.9725 | Pois%: 0.31 | Time: 44.01s
S6 | F1: 0.9998 | ASR: 0.7777 | Pois%: 0.38 | Time: 44.16s
S7 | F1: 0.9997 | ASR: 0.9884 | Pois%: 0.44 | Time: 43.40s
S8 | F1: 0.4885 | ASR: 0.9902 | Pois%: 0.50 | Time: 44.15s
S9 | F1: 0.4713 | ASR: 0.9948 | Pois%: 0.56 | Time: 43.08s
S10 | F1: 0.4559 | ASR: 0.9983 | Pois%: 0.62 | Time: 43.38s
S11 | F1: 0.4469 | ASR: 0.9978 | Pois%: 0.69 | Time: 43.16s
S12 | F1: 0.4373 | ASR: 1.0000 | Pois%: 0.75 | Time: 43.59s
S13 | F1: 0.4373 | ASR: 1.0000 | Pois%: 0.81 | Time: 43.33s
S14 | F1: 0

In [ ]:
# ---------------------------------------------------------
# 📦 INSTALAÇÃO DE DEPENDÊNCIAS
!pip install hdbscan
# ---------------------------------------------------------

import pandas as pd
import numpy as np
import time
import warnings
import os
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import cdist
from google.colab import drive

# Tenta importar HDBSCAN
try:
    import hdbscan
    HAS_HDBSCAN = True
except ImportError:
    HAS_HDBSCAN = False
    print("⚠️ AVISO: Biblioteca 'hdbscan' não encontrada. FLAME será pulado.")

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURAÇÕES
# ==============================================================================
DRIVE_FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/CSAI4CPS/Datasets/Philips_B120N10_Baby_Monitor_gafgyt_attacks_consolidado.csv"

NODES = ["Node1", "Node2", "Node3", "Node4"]
STAGES_ORDER = [f"S{i}" for i in range(17)]
RANDOM_STATE = 42

# Parâmetros Adaptive
ADAPTIVE_MOMENTUM = 0.6
STRICT_PUNISHMENT = 0.1
MIN_TRUST_THRESHOLD = 0.01
FLAME_NOISE_SIGMA = 0.001
HIDDEN_LAYERS = (50, 25)

# ==============================================================================
# 2. UTILITÁRIOS
# ==============================================================================
def get_user_data():
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')
    if not os.path.exists(DRIVE_FILE_PATH):
        print(f"❌ Arquivo não encontrado: {DRIVE_FILE_PATH}")
        return None, None
    try:
        df = pd.read_csv(DRIVE_FILE_PATH)
        print(f"✅ Dataset carregado: {df.shape}")
        return df, "ataque"
    except Exception as e:
        print(f"❌ Erro: {e}")
        return None, None

def generate_stage_data(df_base, feature_cols, target_col, stage_idx, nodes_list):
    alphas_map = {n: 0.0 for n in nodes_list}
    if stage_idx > 0:
        node_idx = (stage_idx - 1) // 4
        step_idx = (stage_idx - 1) % 4
        target_node = nodes_list[min(node_idx, len(nodes_list)-1)]
        current_alpha = [0.25, 0.50, 0.75, 1.00][step_idx]
        for prev_i in range(node_idx):
            alphas_map[nodes_list[prev_i]] = 1.0
        alphas_map[target_node] = current_alpha

    df_shuffled = df_base.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    shards = np.array_split(df_shuffled, len(nodes_list))
    stage_data_list = []

    trigger_feat_idx = -1

    for i, node in enumerate(nodes_list):
        df_node = shards[i].copy()
        y = df_node[target_col].values.astype(int)
        y_poisoned = y.copy()
        X_node = df_node[feature_cols].values.copy()
        alpha = alphas_map[node]

        # Alvo do ataque definido como 1
        if alpha > 0:
            n_poison = int(len(y) * alpha)
            if n_poison > 0:
                poison_idx = np.random.choice(len(y), n_poison, replace=False)
                X_node[poison_idx, trigger_feat_idx] = 99.0
                y_poisoned[poison_idx] = 1

        stage_data_list.append({"node": node, "X": X_node, "y": y_poisoned, "alpha": alpha})
    return stage_data_list

# ==============================================================================
# 3. TREINAMENTO
# ==============================================================================
def create_model(random_state=42):
    return MLPClassifier(hidden_layer_sizes=HIDDEN_LAYERS, activation='relu', solver='adam',
                         alpha=0.0001, batch_size=32, learning_rate_init=0.001,
                         random_state=random_state, max_iter=1)

def get_weights_mlp(model):
    return np.concatenate([w.flatten() for w in model.coefs_] + [b.flatten() for b in model.intercepts_])

def set_weights_mlp(model, flat_weights, ref_model):
    idx = 0
    for i in range(len(model.coefs_)):
        shape = ref_model.coefs_[i].shape
        size = np.prod(shape)
        model.coefs_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size
    for i in range(len(model.intercepts_)):
        shape = ref_model.intercepts_[i].shape
        size = np.prod(shape)
        model.intercepts_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size

def local_training_mlp(global_weights, X, y, classes, ref_model):
    model = create_model(RANDOM_STATE)
    model.partial_fit(X[:5], y[:5], classes=classes)
    set_weights_mlp(model, global_weights.copy(), ref_model)

    for _ in range(5):
        p = np.random.permutation(len(y))
        model.partial_fit(X[p], y[p], classes=classes)
    return get_weights_mlp(model) - global_weights

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    try:
        y_prob = model.predict_proba(X_test)
        loss = log_loss(y_test, y_prob, labels=model.classes_)
    except: loss = 0.0
    return {"acc": accuracy_score(y_test, y_pred), "f1": f1_score(y_test, y_pred, average='macro'), "loss": loss}

# ==============================================================================
# 4. BENCHMARK
# ==============================================================================
def run_benchmark_ultimate(df_raw, target_col):
    print("\n⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive")

    feature_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in feature_cols: feature_cols.remove(target_col)

    X_temp, X_root, y_temp, y_root = train_test_split(df_raw[feature_cols].values, df_raw[target_col].values, test_size=0.05, stratify=df_raw[target_col].values, random_state=RANDOM_STATE)
    X_test, _, y_test, _ = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=RANDOM_STATE)

    scaler = StandardScaler().fit(X_test)
    X_test = scaler.transform(X_test)
    X_root = scaler.transform(X_root)
    X_backdoor = X_test.copy()
    X_backdoor[:, -1] = 99.0
    classes = np.unique(y_test)

    STRATEGIES = ["FedAvg", "Krum", "FLTrust", "Adaptive"]
    if HAS_HDBSCAN: STRATEGIES.insert(3, "FLAME")

    results = []

    for strategy in STRATEGIES:
        print(f"\n🚀 {strategy.upper()} --------------------------------")
        global_model = create_model(RANDOM_STATE)
        global_model.partial_fit(X_test[:100], y_test[:100], classes=classes)
        global_weights = get_weights_mlp(global_model)

        trust_memory = {n: 0.5 for n in NODES}

        for stage_idx, stage_name in enumerate(STAGES_ORDER):
            # --- MEDIÇÃO DE TEMPO (Início) ---
            start_time = time.time()

            stage_data = generate_stage_data(df_raw, feature_cols, target_col, stage_idx, NODES)

            # --- CÁLCULO DA % DE DADOS ATACADOS ---
            # Soma os alphas de cada nó e divide pelo total de nós (média global de veneno)
            total_alpha = sum([d['alpha'] for d in stage_data])
            global_poison_pct = total_alpha / len(NODES)

            server_update = None
            server_norm = 1.0
            if strategy in ["FLTrust", "Adaptive"]:
                server_delta = local_training_mlp(global_weights, X_root, y_root, classes, global_model)
                server_update = server_delta
                server_norm = np.linalg.norm(server_update) + 1e-12

            local_updates = []
            valid_nodes = []
            for node_dict in stage_data:
                X_node = scaler.transform(node_dict['X'])
                delta = local_training_mlp(global_weights, X_node, node_dict['y'], classes, global_model)
                local_updates.append(delta)
                valid_nodes.append(node_dict['node'])

            agg_update = np.zeros_like(global_weights)
            audit_trust = 0.0

            if len(local_updates) > 0:
                if strategy == "FedAvg":
                    agg_update = np.mean(local_updates, axis=0)
                    audit_trust = 1.0

                elif strategy == "Krum":
                    updates_stack = np.stack(local_updates)
                    n = len(local_updates)
                    f = max(1, int(n * 0.25))
                    k = max(1, n - f - 2)
                    dists = cdist(updates_stack, updates_stack, metric='euclidean')
                    scores = []
                    for i in range(n):
                        d_sorted = np.sort(dists[i])
                        scores.append(np.sum(d_sorted[1:k+1]))
                    best_idx = np.argmin(scores)
                    agg_update = updates_stack[best_idx]
                    audit_trust = 1.0 if valid_nodes[best_idx] == "Node1" else 0.0

                elif strategy == "FLTrust":
                    scores = []
                    norm_deltas = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        score = max(0.0, cosine)
                        scores.append(score)
                        norm_deltas.append(delta * (server_norm / norm))
                    if sum(scores) > 0:
                        total = sum(scores)
                        for i, d in enumerate(norm_deltas):
                            agg_update += (scores[i]/total) * d
                        audit_trust = scores[0]

                elif strategy == "FLAME":
                    updates_stack = np.stack(local_updates)
                    dists = cosine_distances(updates_stack)
                    min_size = max(2, int(len(local_updates) * 0.5) + 1)
                    clusterer = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=min_size, allow_single_cluster=True)
                    labels = clusterer.fit_predict(dists)
                    unique, counts = np.unique(labels, return_counts=True)
                    valid_l = unique[unique != -1]
                    if len(valid_l) > 0:
                        best_c = valid_l[np.argmax(counts[unique != -1])]
                        benign_idxs = np.where(labels == best_c)[0]
                    else:
                        benign_idxs = np.arange(len(local_updates))
                    benign_updates = updates_stack[benign_idxs]
                    norms = np.linalg.norm(benign_updates, axis=1)
                    median_norm = np.median(norms)
                    clipped = [u * min(1, median_norm/(np.linalg.norm(u)+1e-12)) for u in benign_updates]
                    if clipped:
                        avg_u = np.mean(clipped, axis=0)
                        noise = np.random.normal(0, FLAME_NOISE_SIGMA * median_norm, avg_u.shape)
                        agg_update = avg_u + noise
                        audit_trust = 1.0 if 0 in benign_idxs else 0.0

                elif strategy == "Adaptive":
                    scores = []
                    valid_ups = []
                    raw_cosines = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        raw_cosines.append(cosine)

                    mu = np.mean(raw_cosines)
                    std = np.std(raw_cosines)
                    dynamic_thresh = max(0.05, mu - (1.0 * std))

                    for i, cosine in enumerate(raw_cosines):
                        node = valid_nodes[i]
                        delta = local_updates[i]
                        norm = np.linalg.norm(delta) + 1e-12
                        if cosine < 0:
                            trust_memory[node] = 0.0
                            new_trust = 0.0
                        else:
                            old = trust_memory[node]
                            if cosine < dynamic_thresh:
                                new_trust = cosine * STRICT_PUNISHMENT
                            else:
                                new_trust = (ADAPTIVE_MOMENTUM * old) + ((1 - ADAPTIVE_MOMENTUM) * cosine)
                            trust_memory[node] = new_trust
                        if new_trust > MIN_TRUST_THRESHOLD:
                            scores.append(new_trust)
                            valid_ups.append(delta * (server_norm / norm))
                        if node == "Node1": audit_trust = new_trust

                    if scores:
                        total = sum(scores)
                        for i, d in enumerate(valid_ups):
                            agg_update += (scores[i]/total) * d

            global_weights += agg_update
            set_weights_mlp(global_model, global_weights, global_model)

            # --- MEDIÇÃO DE TEMPO (Fim) ---
            end_time = time.time()
            elapsed_time = end_time - start_time

            metrics = evaluate(global_model, X_test, y_test)
            pred_bd = global_model.predict(X_backdoor)
            asr = np.mean(pred_bd == 1)

            print(f"{stage_name} | F1: {metrics['f1']:.4f} | ASR: {asr:.4f} | Pois%: {global_poison_pct:.2f} | Time: {elapsed_time:.2f}s")

            results.append({
                "Strategy": strategy,
                "Stage": stage_name,
                "F1_Main": metrics['f1'],
                "ASR_Backdoor": asr,
                "Global_Poisoning_Rate": global_poison_pct, # <--- Nova Coluna
                "Time_Seconds": elapsed_time,               # <--- Nova Coluna
                "Accuracy": metrics['acc'],
                "Trust_N1": audit_trust
            })
            gc.collect()

    return pd.DataFrame(results)

if __name__ == "__main__":
    start_global_time = time.time()

    df_raw, target_col = get_user_data()
    if df_raw is not None:
        if target_col not in df_raw.columns:
            target_col = input("👉 Coluna de Label (ex: attack): ").strip()

        df_res = run_benchmark_ultimate(df_raw, target_col)

        # --- NOMEAÇÃO AUTOMÁTICA COM ARQUIVO DE ENTRADA ---
        # Extrai o nome do arquivo (ex: "Ecobee_Thermostat_mirai_attacks_consolidado")
        input_filename = os.path.basename(DRIVE_FILE_PATH).replace(".csv", "")
        # Monta o nome de saída
        fname = f"{input_filename}_RESULTADOS_ULTIMATE_5_METHODS.csv"

        # Salva localmente e no drive
        dir_path = os.path.dirname(DRIVE_FILE_PATH)

        df_res.to_csv(fname, index=False)
        print(f"\n💾 Salvo Local: {fname}")

        try:
            full_path = os.path.join(dir_path, fname)
            df_res.to_csv(full_path, index=False)
            print(f"✅ Salvo Drive: {full_path}")
        except: pass

    end_global_time = time.time()
    print(f"\n⏱️ Tempo Total de Execução: {(end_global_time - start_global_time)/60:.2f} minutos")

✅ Dataset carregado: (487963, 117)

⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive

🚀 FEDAVG --------------------------------
S0 | F1: 0.4565 | ASR: 0.0000 | Pois%: 0.00 | Time: 26.98s
S1 | F1: 0.4565 | ASR: 0.0000 | Pois%: 0.06 | Time: 26.47s
S2 | F1: 0.4565 | ASR: 0.0000 | Pois%: 0.12 | Time: 26.18s
S3 | F1: 0.4565 | ASR: 0.0000 | Pois%: 0.19 | Time: 27.02s
S4 | F1: 0.5489 | ASR: 0.2889 | Pois%: 0.25 | Time: 26.25s
S5 | F1: 0.5034 | ASR: 0.0049 | Pois%: 0.31 | Time: 26.93s
S6 | F1: 0.4792 | ASR: 0.0008 | Pois%: 0.38 | Time: 26.67s
S7 | F1: 0.5615 | ASR: 0.1806 | Pois%: 0.44 | Time: 27.19s
S8 | F1: 0.1403 | ASR: 0.2911 | Pois%: 0.50 | Time: 26.60s
S9 | F1: 0.1382 | ASR: 0.1313 | Pois%: 0.56 | Time: 26.61s
S10 | F1: 0.1381 | ASR: 0.2950 | Pois%: 0.62 | Time: 26.45s
S11 | F1: 0.1381 | ASR: 1.0000 | Pois%: 0.69 | Time: 26.44s
S12 | F1: 0.1381 | ASR: 1.0000 | Pois%: 0.75 | Time: 26.48s
S13 | F1: 0.1381 | ASR: 1.0000 | Pois%: 0.81 | Time: 26.74s
S14 | F1: 0

In [ ]:
# ---------------------------------------------------------
# 📦 INSTALAÇÃO DE DEPENDÊNCIAS
!pip install hdbscan
# ---------------------------------------------------------

import pandas as pd
import numpy as np
import time
import warnings
import os
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import cdist
from google.colab import drive

# Tenta importar HDBSCAN
try:
    import hdbscan
    HAS_HDBSCAN = True
except ImportError:
    HAS_HDBSCAN = False
    print("⚠️ AVISO: Biblioteca 'hdbscan' não encontrada. FLAME será pulado.")

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURAÇÕES
# ==============================================================================
DRIVE_FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/CSAI4CPS/Datasets/Ennio_Doorbell_gafgyt_attacks_consolidado.csv"

NODES = ["Node1", "Node2", "Node3", "Node4"]
STAGES_ORDER = [f"S{i}" for i in range(17)]
RANDOM_STATE = 42

# Parâmetros Adaptive
ADAPTIVE_MOMENTUM = 0.6
STRICT_PUNISHMENT = 0.1
MIN_TRUST_THRESHOLD = 0.01
FLAME_NOISE_SIGMA = 0.001
HIDDEN_LAYERS = (50, 25)

# ==============================================================================
# 2. UTILITÁRIOS
# ==============================================================================
def get_user_data():
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')
    if not os.path.exists(DRIVE_FILE_PATH):
        print(f"❌ Arquivo não encontrado: {DRIVE_FILE_PATH}")
        return None, None
    try:
        df = pd.read_csv(DRIVE_FILE_PATH)
        print(f"✅ Dataset carregado: {df.shape}")
        return df, "ataque"
    except Exception as e:
        print(f"❌ Erro: {e}")
        return None, None

def generate_stage_data(df_base, feature_cols, target_col, stage_idx, nodes_list):
    alphas_map = {n: 0.0 for n in nodes_list}
    if stage_idx > 0:
        node_idx = (stage_idx - 1) // 4
        step_idx = (stage_idx - 1) % 4
        target_node = nodes_list[min(node_idx, len(nodes_list)-1)]
        current_alpha = [0.25, 0.50, 0.75, 1.00][step_idx]
        for prev_i in range(node_idx):
            alphas_map[nodes_list[prev_i]] = 1.0
        alphas_map[target_node] = current_alpha

    df_shuffled = df_base.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    shards = np.array_split(df_shuffled, len(nodes_list))
    stage_data_list = []

    trigger_feat_idx = -1

    for i, node in enumerate(nodes_list):
        df_node = shards[i].copy()
        y = df_node[target_col].values.astype(int)
        y_poisoned = y.copy()
        X_node = df_node[feature_cols].values.copy()
        alpha = alphas_map[node]

        # Alvo do ataque definido como 1
        if alpha > 0:
            n_poison = int(len(y) * alpha)
            if n_poison > 0:
                poison_idx = np.random.choice(len(y), n_poison, replace=False)
                X_node[poison_idx, trigger_feat_idx] = 99.0
                y_poisoned[poison_idx] = 1

        stage_data_list.append({"node": node, "X": X_node, "y": y_poisoned, "alpha": alpha})
    return stage_data_list

# ==============================================================================
# 3. TREINAMENTO
# ==============================================================================
def create_model(random_state=42):
    return MLPClassifier(hidden_layer_sizes=HIDDEN_LAYERS, activation='relu', solver='adam',
                         alpha=0.0001, batch_size=32, learning_rate_init=0.001,
                         random_state=random_state, max_iter=1)

def get_weights_mlp(model):
    return np.concatenate([w.flatten() for w in model.coefs_] + [b.flatten() for b in model.intercepts_])

def set_weights_mlp(model, flat_weights, ref_model):
    idx = 0
    for i in range(len(model.coefs_)):
        shape = ref_model.coefs_[i].shape
        size = np.prod(shape)
        model.coefs_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size
    for i in range(len(model.intercepts_)):
        shape = ref_model.intercepts_[i].shape
        size = np.prod(shape)
        model.intercepts_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size

def local_training_mlp(global_weights, X, y, classes, ref_model):
    model = create_model(RANDOM_STATE)
    model.partial_fit(X[:5], y[:5], classes=classes)
    set_weights_mlp(model, global_weights.copy(), ref_model)

    for _ in range(5):
        p = np.random.permutation(len(y))
        model.partial_fit(X[p], y[p], classes=classes)
    return get_weights_mlp(model) - global_weights

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    try:
        y_prob = model.predict_proba(X_test)
        loss = log_loss(y_test, y_prob, labels=model.classes_)
    except: loss = 0.0
    return {"acc": accuracy_score(y_test, y_pred), "f1": f1_score(y_test, y_pred, average='macro'), "loss": loss}

# ==============================================================================
# 4. BENCHMARK
# ==============================================================================
def run_benchmark_ultimate(df_raw, target_col):
    print("\n⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive")

    feature_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in feature_cols: feature_cols.remove(target_col)

    X_temp, X_root, y_temp, y_root = train_test_split(df_raw[feature_cols].values, df_raw[target_col].values, test_size=0.05, stratify=df_raw[target_col].values, random_state=RANDOM_STATE)
    X_test, _, y_test, _ = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=RANDOM_STATE)

    scaler = StandardScaler().fit(X_test)
    X_test = scaler.transform(X_test)
    X_root = scaler.transform(X_root)
    X_backdoor = X_test.copy()
    X_backdoor[:, -1] = 99.0
    classes = np.unique(y_test)

    STRATEGIES = ["FedAvg", "Krum", "FLTrust", "Adaptive"]
    if HAS_HDBSCAN: STRATEGIES.insert(3, "FLAME")

    results = []

    for strategy in STRATEGIES:
        print(f"\n🚀 {strategy.upper()} --------------------------------")
        global_model = create_model(RANDOM_STATE)
        global_model.partial_fit(X_test[:100], y_test[:100], classes=classes)
        global_weights = get_weights_mlp(global_model)

        trust_memory = {n: 0.5 for n in NODES}

        for stage_idx, stage_name in enumerate(STAGES_ORDER):
            # --- MEDIÇÃO DE TEMPO (Início) ---
            start_time = time.time()

            stage_data = generate_stage_data(df_raw, feature_cols, target_col, stage_idx, NODES)

            # --- CÁLCULO DA % DE DADOS ATACADOS ---
            # Soma os alphas de cada nó e divide pelo total de nós (média global de veneno)
            total_alpha = sum([d['alpha'] for d in stage_data])
            global_poison_pct = total_alpha / len(NODES)

            server_update = None
            server_norm = 1.0
            if strategy in ["FLTrust", "Adaptive"]:
                server_delta = local_training_mlp(global_weights, X_root, y_root, classes, global_model)
                server_update = server_delta
                server_norm = np.linalg.norm(server_update) + 1e-12

            local_updates = []
            valid_nodes = []
            for node_dict in stage_data:
                X_node = scaler.transform(node_dict['X'])
                delta = local_training_mlp(global_weights, X_node, node_dict['y'], classes, global_model)
                local_updates.append(delta)
                valid_nodes.append(node_dict['node'])

            agg_update = np.zeros_like(global_weights)
            audit_trust = 0.0

            if len(local_updates) > 0:
                if strategy == "FedAvg":
                    agg_update = np.mean(local_updates, axis=0)
                    audit_trust = 1.0

                elif strategy == "Krum":
                    updates_stack = np.stack(local_updates)
                    n = len(local_updates)
                    f = max(1, int(n * 0.25))
                    k = max(1, n - f - 2)
                    dists = cdist(updates_stack, updates_stack, metric='euclidean')
                    scores = []
                    for i in range(n):
                        d_sorted = np.sort(dists[i])
                        scores.append(np.sum(d_sorted[1:k+1]))
                    best_idx = np.argmin(scores)
                    agg_update = updates_stack[best_idx]
                    audit_trust = 1.0 if valid_nodes[best_idx] == "Node1" else 0.0

                elif strategy == "FLTrust":
                    scores = []
                    norm_deltas = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        score = max(0.0, cosine)
                        scores.append(score)
                        norm_deltas.append(delta * (server_norm / norm))
                    if sum(scores) > 0:
                        total = sum(scores)
                        for i, d in enumerate(norm_deltas):
                            agg_update += (scores[i]/total) * d
                        audit_trust = scores[0]

                elif strategy == "FLAME":
                    updates_stack = np.stack(local_updates)
                    dists = cosine_distances(updates_stack)
                    min_size = max(2, int(len(local_updates) * 0.5) + 1)
                    clusterer = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=min_size, allow_single_cluster=True)
                    labels = clusterer.fit_predict(dists)
                    unique, counts = np.unique(labels, return_counts=True)
                    valid_l = unique[unique != -1]
                    if len(valid_l) > 0:
                        best_c = valid_l[np.argmax(counts[unique != -1])]
                        benign_idxs = np.where(labels == best_c)[0]
                    else:
                        benign_idxs = np.arange(len(local_updates))
                    benign_updates = updates_stack[benign_idxs]
                    norms = np.linalg.norm(benign_updates, axis=1)
                    median_norm = np.median(norms)
                    clipped = [u * min(1, median_norm/(np.linalg.norm(u)+1e-12)) for u in benign_updates]
                    if clipped:
                        avg_u = np.mean(clipped, axis=0)
                        noise = np.random.normal(0, FLAME_NOISE_SIGMA * median_norm, avg_u.shape)
                        agg_update = avg_u + noise
                        audit_trust = 1.0 if 0 in benign_idxs else 0.0

                elif strategy == "Adaptive":
                    scores = []
                    valid_ups = []
                    raw_cosines = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        raw_cosines.append(cosine)

                    mu = np.mean(raw_cosines)
                    std = np.std(raw_cosines)
                    dynamic_thresh = max(0.05, mu - (1.0 * std))

                    for i, cosine in enumerate(raw_cosines):
                        node = valid_nodes[i]
                        delta = local_updates[i]
                        norm = np.linalg.norm(delta) + 1e-12
                        if cosine < 0:
                            trust_memory[node] = 0.0
                            new_trust = 0.0
                        else:
                            old = trust_memory[node]
                            if cosine < dynamic_thresh:
                                new_trust = cosine * STRICT_PUNISHMENT
                            else:
                                new_trust = (ADAPTIVE_MOMENTUM * old) + ((1 - ADAPTIVE_MOMENTUM) * cosine)
                            trust_memory[node] = new_trust
                        if new_trust > MIN_TRUST_THRESHOLD:
                            scores.append(new_trust)
                            valid_ups.append(delta * (server_norm / norm))
                        if node == "Node1": audit_trust = new_trust

                    if scores:
                        total = sum(scores)
                        for i, d in enumerate(valid_ups):
                            agg_update += (scores[i]/total) * d

            global_weights += agg_update
            set_weights_mlp(global_model, global_weights, global_model)

            # --- MEDIÇÃO DE TEMPO (Fim) ---
            end_time = time.time()
            elapsed_time = end_time - start_time

            metrics = evaluate(global_model, X_test, y_test)
            pred_bd = global_model.predict(X_backdoor)
            asr = np.mean(pred_bd == 1)

            print(f"{stage_name} | F1: {metrics['f1']:.4f} | ASR: {asr:.4f} | Pois%: {global_poison_pct:.2f} | Time: {elapsed_time:.2f}s")

            results.append({
                "Strategy": strategy,
                "Stage": stage_name,
                "F1_Main": metrics['f1'],
                "ASR_Backdoor": asr,
                "Global_Poisoning_Rate": global_poison_pct, # <--- Nova Coluna
                "Time_Seconds": elapsed_time,               # <--- Nova Coluna
                "Accuracy": metrics['acc'],
                "Trust_N1": audit_trust
            })
            gc.collect()

    return pd.DataFrame(results)

if __name__ == "__main__":
    start_global_time = time.time()

    df_raw, target_col = get_user_data()
    if df_raw is not None:
        if target_col not in df_raw.columns:
            target_col = input("👉 Coluna de Label (ex: attack): ").strip()

        df_res = run_benchmark_ultimate(df_raw, target_col)

        # --- NOMEAÇÃO AUTOMÁTICA COM ARQUIVO DE ENTRADA ---
        # Extrai o nome do arquivo (ex: "Ecobee_Thermostat_mirai_attacks_consolidado")
        input_filename = os.path.basename(DRIVE_FILE_PATH).replace(".csv", "")
        # Monta o nome de saída
        fname = f"{input_filename}_RESULTADOS_ULTIMATE_5_METHODS.csv"

        # Salva localmente e no drive
        dir_path = os.path.dirname(DRIVE_FILE_PATH)

        df_res.to_csv(fname, index=False)
        print(f"\n💾 Salvo Local: {fname}")

        try:
            full_path = os.path.join(dir_path, fname)
            df_res.to_csv(full_path, index=False)
            print(f"✅ Salvo Drive: {full_path}")
        except: pass

    end_global_time = time.time()
    print(f"\n⏱️ Tempo Total de Execução: {(end_global_time - start_global_time)/60:.2f} minutos")

✅ Dataset carregado: (355500, 117)

⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive

🚀 FEDAVG --------------------------------
S0 | F1: 0.6755 | ASR: 0.9998 | Pois%: 0.00 | Time: 19.39s
S1 | F1: 0.6649 | ASR: 0.2819 | Pois%: 0.06 | Time: 19.38s
S2 | F1: 0.4005 | ASR: 0.3989 | Pois%: 0.12 | Time: 19.31s
S3 | F1: 0.4004 | ASR: 0.3992 | Pois%: 0.19 | Time: 19.42s
S4 | F1: 0.4004 | ASR: 0.3838 | Pois%: 0.25 | Time: 19.48s
S5 | F1: 0.4004 | ASR: 0.2029 | Pois%: 0.31 | Time: 19.26s
S6 | F1: 0.4004 | ASR: 0.2705 | Pois%: 0.38 | Time: 19.17s
S7 | F1: 0.4004 | ASR: 0.1511 | Pois%: 0.44 | Time: 19.23s
S8 | F1: 0.4004 | ASR: 0.3808 | Pois%: 0.50 | Time: 19.25s
S9 | F1: 0.4004 | ASR: 1.0000 | Pois%: 0.56 | Time: 19.67s
S10 | F1: 0.4003 | ASR: 1.0000 | Pois%: 0.62 | Time: 19.53s
S11 | F1: 0.4003 | ASR: 1.0000 | Pois%: 0.69 | Time: 19.24s
S12 | F1: 0.4003 | ASR: 1.0000 | Pois%: 0.75 | Time: 19.36s
S13 | F1: 0.4003 | ASR: 1.0000 | Pois%: 0.81 | Time: 19.55s
S14 | F1: 0

In [ ]:
# ---------------------------------------------------------
# 📦 INSTALAÇÃO DE DEPENDÊNCIAS
!pip install hdbscan
# ---------------------------------------------------------

import pandas as pd
import numpy as np
import time
import warnings
import os
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import cdist
from google.colab import drive

# Tenta importar HDBSCAN
try:
    import hdbscan
    HAS_HDBSCAN = True
except ImportError:
    HAS_HDBSCAN = False
    print("⚠️ AVISO: Biblioteca 'hdbscan' não encontrada. FLAME será pulado.")

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURAÇÕES
# ==============================================================================
DRIVE_FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/CSAI4CPS/Datasets/Ecobee_Thermostat_gafgyt_attacks_consolidado.csv"

NODES = ["Node1", "Node2", "Node3", "Node4"]
STAGES_ORDER = [f"S{i}" for i in range(17)]
RANDOM_STATE = 42

# Parâmetros Adaptive
ADAPTIVE_MOMENTUM = 0.6
STRICT_PUNISHMENT = 0.1
MIN_TRUST_THRESHOLD = 0.01
FLAME_NOISE_SIGMA = 0.001
HIDDEN_LAYERS = (50, 25)

# ==============================================================================
# 2. UTILITÁRIOS
# ==============================================================================
def get_user_data():
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')
    if not os.path.exists(DRIVE_FILE_PATH):
        print(f"❌ Arquivo não encontrado: {DRIVE_FILE_PATH}")
        return None, None
    try:
        df = pd.read_csv(DRIVE_FILE_PATH)
        print(f"✅ Dataset carregado: {df.shape}")
        return df, "ataque"
    except Exception as e:
        print(f"❌ Erro: {e}")
        return None, None

def generate_stage_data(df_base, feature_cols, target_col, stage_idx, nodes_list):
    alphas_map = {n: 0.0 for n in nodes_list}
    if stage_idx > 0:
        node_idx = (stage_idx - 1) // 4
        step_idx = (stage_idx - 1) % 4
        target_node = nodes_list[min(node_idx, len(nodes_list)-1)]
        current_alpha = [0.25, 0.50, 0.75, 1.00][step_idx]
        for prev_i in range(node_idx):
            alphas_map[nodes_list[prev_i]] = 1.0
        alphas_map[target_node] = current_alpha

    df_shuffled = df_base.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    shards = np.array_split(df_shuffled, len(nodes_list))
    stage_data_list = []

    trigger_feat_idx = -1

    for i, node in enumerate(nodes_list):
        df_node = shards[i].copy()
        y = df_node[target_col].values.astype(int)
        y_poisoned = y.copy()
        X_node = df_node[feature_cols].values.copy()
        alpha = alphas_map[node]

        # Alvo do ataque definido como 1
        if alpha > 0:
            n_poison = int(len(y) * alpha)
            if n_poison > 0:
                poison_idx = np.random.choice(len(y), n_poison, replace=False)
                X_node[poison_idx, trigger_feat_idx] = 99.0
                y_poisoned[poison_idx] = 1

        stage_data_list.append({"node": node, "X": X_node, "y": y_poisoned, "alpha": alpha})
    return stage_data_list

# ==============================================================================
# 3. TREINAMENTO
# ==============================================================================
def create_model(random_state=42):
    return MLPClassifier(hidden_layer_sizes=HIDDEN_LAYERS, activation='relu', solver='adam',
                         alpha=0.0001, batch_size=32, learning_rate_init=0.001,
                         random_state=random_state, max_iter=1)

def get_weights_mlp(model):
    return np.concatenate([w.flatten() for w in model.coefs_] + [b.flatten() for b in model.intercepts_])

def set_weights_mlp(model, flat_weights, ref_model):
    idx = 0
    for i in range(len(model.coefs_)):
        shape = ref_model.coefs_[i].shape
        size = np.prod(shape)
        model.coefs_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size
    for i in range(len(model.intercepts_)):
        shape = ref_model.intercepts_[i].shape
        size = np.prod(shape)
        model.intercepts_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size

def local_training_mlp(global_weights, X, y, classes, ref_model):
    model = create_model(RANDOM_STATE)
    model.partial_fit(X[:5], y[:5], classes=classes)
    set_weights_mlp(model, global_weights.copy(), ref_model)

    for _ in range(5):
        p = np.random.permutation(len(y))
        model.partial_fit(X[p], y[p], classes=classes)
    return get_weights_mlp(model) - global_weights

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    try:
        y_prob = model.predict_proba(X_test)
        loss = log_loss(y_test, y_prob, labels=model.classes_)
    except: loss = 0.0
    return {"acc": accuracy_score(y_test, y_pred), "f1": f1_score(y_test, y_pred, average='macro'), "loss": loss}

# ==============================================================================
# 4. BENCHMARK
# ==============================================================================
def run_benchmark_ultimate(df_raw, target_col):
    print("\n⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive")

    feature_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in feature_cols: feature_cols.remove(target_col)

    X_temp, X_root, y_temp, y_root = train_test_split(df_raw[feature_cols].values, df_raw[target_col].values, test_size=0.05, stratify=df_raw[target_col].values, random_state=RANDOM_STATE)
    X_test, _, y_test, _ = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=RANDOM_STATE)

    scaler = StandardScaler().fit(X_test)
    X_test = scaler.transform(X_test)
    X_root = scaler.transform(X_root)
    X_backdoor = X_test.copy()
    X_backdoor[:, -1] = 99.0
    classes = np.unique(y_test)

    STRATEGIES = ["FedAvg", "Krum", "FLTrust", "Adaptive"]
    if HAS_HDBSCAN: STRATEGIES.insert(3, "FLAME")

    results = []

    for strategy in STRATEGIES:
        print(f"\n🚀 {strategy.upper()} --------------------------------")
        global_model = create_model(RANDOM_STATE)
        global_model.partial_fit(X_test[:100], y_test[:100], classes=classes)
        global_weights = get_weights_mlp(global_model)

        trust_memory = {n: 0.5 for n in NODES}

        for stage_idx, stage_name in enumerate(STAGES_ORDER):
            # --- MEDIÇÃO DE TEMPO (Início) ---
            start_time = time.time()

            stage_data = generate_stage_data(df_raw, feature_cols, target_col, stage_idx, NODES)

            # --- CÁLCULO DA % DE DADOS ATACADOS ---
            # Soma os alphas de cada nó e divide pelo total de nós (média global de veneno)
            total_alpha = sum([d['alpha'] for d in stage_data])
            global_poison_pct = total_alpha / len(NODES)

            server_update = None
            server_norm = 1.0
            if strategy in ["FLTrust", "Adaptive"]:
                server_delta = local_training_mlp(global_weights, X_root, y_root, classes, global_model)
                server_update = server_delta
                server_norm = np.linalg.norm(server_update) + 1e-12

            local_updates = []
            valid_nodes = []
            for node_dict in stage_data:
                X_node = scaler.transform(node_dict['X'])
                delta = local_training_mlp(global_weights, X_node, node_dict['y'], classes, global_model)
                local_updates.append(delta)
                valid_nodes.append(node_dict['node'])

            agg_update = np.zeros_like(global_weights)
            audit_trust = 0.0

            if len(local_updates) > 0:
                if strategy == "FedAvg":
                    agg_update = np.mean(local_updates, axis=0)
                    audit_trust = 1.0

                elif strategy == "Krum":
                    updates_stack = np.stack(local_updates)
                    n = len(local_updates)
                    f = max(1, int(n * 0.25))
                    k = max(1, n - f - 2)
                    dists = cdist(updates_stack, updates_stack, metric='euclidean')
                    scores = []
                    for i in range(n):
                        d_sorted = np.sort(dists[i])
                        scores.append(np.sum(d_sorted[1:k+1]))
                    best_idx = np.argmin(scores)
                    agg_update = updates_stack[best_idx]
                    audit_trust = 1.0 if valid_nodes[best_idx] == "Node1" else 0.0

                elif strategy == "FLTrust":
                    scores = []
                    norm_deltas = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        score = max(0.0, cosine)
                        scores.append(score)
                        norm_deltas.append(delta * (server_norm / norm))
                    if sum(scores) > 0:
                        total = sum(scores)
                        for i, d in enumerate(norm_deltas):
                            agg_update += (scores[i]/total) * d
                        audit_trust = scores[0]

                elif strategy == "FLAME":
                    updates_stack = np.stack(local_updates)
                    dists = cosine_distances(updates_stack)
                    min_size = max(2, int(len(local_updates) * 0.5) + 1)
                    clusterer = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=min_size, allow_single_cluster=True)
                    labels = clusterer.fit_predict(dists)
                    unique, counts = np.unique(labels, return_counts=True)
                    valid_l = unique[unique != -1]
                    if len(valid_l) > 0:
                        best_c = valid_l[np.argmax(counts[unique != -1])]
                        benign_idxs = np.where(labels == best_c)[0]
                    else:
                        benign_idxs = np.arange(len(local_updates))
                    benign_updates = updates_stack[benign_idxs]
                    norms = np.linalg.norm(benign_updates, axis=1)
                    median_norm = np.median(norms)
                    clipped = [u * min(1, median_norm/(np.linalg.norm(u)+1e-12)) for u in benign_updates]
                    if clipped:
                        avg_u = np.mean(clipped, axis=0)
                        noise = np.random.normal(0, FLAME_NOISE_SIGMA * median_norm, avg_u.shape)
                        agg_update = avg_u + noise
                        audit_trust = 1.0 if 0 in benign_idxs else 0.0

                elif strategy == "Adaptive":
                    scores = []
                    valid_ups = []
                    raw_cosines = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        raw_cosines.append(cosine)

                    mu = np.mean(raw_cosines)
                    std = np.std(raw_cosines)
                    dynamic_thresh = max(0.05, mu - (1.0 * std))

                    for i, cosine in enumerate(raw_cosines):
                        node = valid_nodes[i]
                        delta = local_updates[i]
                        norm = np.linalg.norm(delta) + 1e-12
                        if cosine < 0:
                            trust_memory[node] = 0.0
                            new_trust = 0.0
                        else:
                            old = trust_memory[node]
                            if cosine < dynamic_thresh:
                                new_trust = cosine * STRICT_PUNISHMENT
                            else:
                                new_trust = (ADAPTIVE_MOMENTUM * old) + ((1 - ADAPTIVE_MOMENTUM) * cosine)
                            trust_memory[node] = new_trust
                        if new_trust > MIN_TRUST_THRESHOLD:
                            scores.append(new_trust)
                            valid_ups.append(delta * (server_norm / norm))
                        if node == "Node1": audit_trust = new_trust

                    if scores:
                        total = sum(scores)
                        for i, d in enumerate(valid_ups):
                            agg_update += (scores[i]/total) * d

            global_weights += agg_update
            set_weights_mlp(global_model, global_weights, global_model)

            # --- MEDIÇÃO DE TEMPO (Fim) ---
            end_time = time.time()
            elapsed_time = end_time - start_time

            metrics = evaluate(global_model, X_test, y_test)
            pred_bd = global_model.predict(X_backdoor)
            asr = np.mean(pred_bd == 1)

            print(f"{stage_name} | F1: {metrics['f1']:.4f} | ASR: {asr:.4f} | Pois%: {global_poison_pct:.2f} | Time: {elapsed_time:.2f}s")

            results.append({
                "Strategy": strategy,
                "Stage": stage_name,
                "F1_Main": metrics['f1'],
                "ASR_Backdoor": asr,
                "Global_Poisoning_Rate": global_poison_pct, # <--- Nova Coluna
                "Time_Seconds": elapsed_time,               # <--- Nova Coluna
                "Accuracy": metrics['acc'],
                "Trust_N1": audit_trust
            })
            gc.collect()

    return pd.DataFrame(results)

if __name__ == "__main__":
    start_global_time = time.time()

    df_raw, target_col = get_user_data()
    if df_raw is not None:
        if target_col not in df_raw.columns:
            target_col = input("👉 Coluna de Label (ex: attack): ").strip()

        df_res = run_benchmark_ultimate(df_raw, target_col)

        # --- NOMEAÇÃO AUTOMÁTICA COM ARQUIVO DE ENTRADA ---
        # Extrai o nome do arquivo (ex: "Ecobee_Thermostat_mirai_attacks_consolidado")
        input_filename = os.path.basename(DRIVE_FILE_PATH).replace(".csv", "")
        # Monta o nome de saída
        fname = f"{input_filename}_RESULTADOS_ULTIMATE_5_METHODS.csv"

        # Salva localmente e no drive
        dir_path = os.path.dirname(DRIVE_FILE_PATH)

        df_res.to_csv(fname, index=False)
        print(f"\n💾 Salvo Local: {fname}")

        try:
            full_path = os.path.join(dir_path, fname)
            df_res.to_csv(full_path, index=False)
            print(f"✅ Salvo Drive: {full_path}")
        except: pass

    end_global_time = time.time()
    print(f"\n⏱️ Tempo Total de Execução: {(end_global_time - start_global_time)/60:.2f} minutos")

✅ Dataset carregado: (323743, 117)

⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive

🚀 FEDAVG --------------------------------
S0 | F1: 0.7050 | ASR: 0.0002 | Pois%: 0.00 | Time: 17.58s
S1 | F1: 0.7052 | ASR: 0.0001 | Pois%: 0.06 | Time: 17.47s
S2 | F1: 0.7051 | ASR: 0.0001 | Pois%: 0.12 | Time: 17.56s
S3 | F1: 0.7050 | ASR: 0.6741 | Pois%: 0.19 | Time: 17.60s
S4 | F1: 0.6806 | ASR: 0.7572 | Pois%: 0.25 | Time: 17.71s
S5 | F1: 0.6858 | ASR: 0.6750 | Pois%: 0.31 | Time: 17.66s
S6 | F1: 0.4998 | ASR: 0.7494 | Pois%: 0.38 | Time: 17.56s
S7 | F1: 0.3295 | ASR: 0.9600 | Pois%: 0.44 | Time: 17.64s
S8 | F1: 0.3299 | ASR: 0.9895 | Pois%: 0.50 | Time: 17.46s
S9 | F1: 0.3289 | ASR: 1.0000 | Pois%: 0.56 | Time: 17.81s
S10 | F1: 0.3291 | ASR: 0.9997 | Pois%: 0.62 | Time: 17.99s
S11 | F1: 0.3289 | ASR: 1.0000 | Pois%: 0.69 | Time: 17.65s
S12 | F1: 0.3289 | ASR: 1.0000 | Pois%: 0.75 | Time: 17.83s
S13 | F1: 0.3289 | ASR: 1.0000 | Pois%: 0.81 | Time: 17.47s
S14 | F1: 0

In [ ]:
# ---------------------------------------------------------
# 📦 INSTALAÇÃO DE DEPENDÊNCIAS
!pip install hdbscan
# ---------------------------------------------------------

import pandas as pd
import numpy as np
import time
import warnings
import os
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import cdist
from google.colab import drive

# Tenta importar HDBSCAN
try:
    import hdbscan
    HAS_HDBSCAN = True
except ImportError:
    HAS_HDBSCAN = False
    print("⚠️ AVISO: Biblioteca 'hdbscan' não encontrada. FLAME será pulado.")

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURAÇÕES
# ==============================================================================
DRIVE_FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/CSAI4CPS/Datasets/Danmini_Doorbell_mirai_attacks_consolidado.csv"

NODES = ["Node1", "Node2", "Node3", "Node4"]
STAGES_ORDER = [f"S{i}" for i in range(17)]
RANDOM_STATE = 42

# Parâmetros Adaptive
ADAPTIVE_MOMENTUM = 0.6
STRICT_PUNISHMENT = 0.1
MIN_TRUST_THRESHOLD = 0.01
FLAME_NOISE_SIGMA = 0.001
HIDDEN_LAYERS = (50, 25)

# ==============================================================================
# 2. UTILITÁRIOS
# ==============================================================================
def get_user_data():
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')
    if not os.path.exists(DRIVE_FILE_PATH):
        print(f"❌ Arquivo não encontrado: {DRIVE_FILE_PATH}")
        return None, None
    try:
        df = pd.read_csv(DRIVE_FILE_PATH)
        print(f"✅ Dataset carregado: {df.shape}")
        return df, "ataque"
    except Exception as e:
        print(f"❌ Erro: {e}")
        return None, None

def generate_stage_data(df_base, feature_cols, target_col, stage_idx, nodes_list):
    alphas_map = {n: 0.0 for n in nodes_list}
    if stage_idx > 0:
        node_idx = (stage_idx - 1) // 4
        step_idx = (stage_idx - 1) % 4
        target_node = nodes_list[min(node_idx, len(nodes_list)-1)]
        current_alpha = [0.25, 0.50, 0.75, 1.00][step_idx]
        for prev_i in range(node_idx):
            alphas_map[nodes_list[prev_i]] = 1.0
        alphas_map[target_node] = current_alpha

    df_shuffled = df_base.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    shards = np.array_split(df_shuffled, len(nodes_list))
    stage_data_list = []

    trigger_feat_idx = -1

    for i, node in enumerate(nodes_list):
        df_node = shards[i].copy()
        y = df_node[target_col].values.astype(int)
        y_poisoned = y.copy()
        X_node = df_node[feature_cols].values.copy()
        alpha = alphas_map[node]

        # Alvo do ataque definido como 1
        if alpha > 0:
            n_poison = int(len(y) * alpha)
            if n_poison > 0:
                poison_idx = np.random.choice(len(y), n_poison, replace=False)
                X_node[poison_idx, trigger_feat_idx] = 99.0
                y_poisoned[poison_idx] = 1

        stage_data_list.append({"node": node, "X": X_node, "y": y_poisoned, "alpha": alpha})
    return stage_data_list

# ==============================================================================
# 3. TREINAMENTO
# ==============================================================================
def create_model(random_state=42):
    return MLPClassifier(hidden_layer_sizes=HIDDEN_LAYERS, activation='relu', solver='adam',
                         alpha=0.0001, batch_size=32, learning_rate_init=0.001,
                         random_state=random_state, max_iter=1)

def get_weights_mlp(model):
    return np.concatenate([w.flatten() for w in model.coefs_] + [b.flatten() for b in model.intercepts_])

def set_weights_mlp(model, flat_weights, ref_model):
    idx = 0
    for i in range(len(model.coefs_)):
        shape = ref_model.coefs_[i].shape
        size = np.prod(shape)
        model.coefs_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size
    for i in range(len(model.intercepts_)):
        shape = ref_model.intercepts_[i].shape
        size = np.prod(shape)
        model.intercepts_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size

def local_training_mlp(global_weights, X, y, classes, ref_model):
    model = create_model(RANDOM_STATE)
    model.partial_fit(X[:5], y[:5], classes=classes)
    set_weights_mlp(model, global_weights.copy(), ref_model)

    for _ in range(5):
        p = np.random.permutation(len(y))
        model.partial_fit(X[p], y[p], classes=classes)
    return get_weights_mlp(model) - global_weights

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    try:
        y_prob = model.predict_proba(X_test)
        loss = log_loss(y_test, y_prob, labels=model.classes_)
    except: loss = 0.0
    return {"acc": accuracy_score(y_test, y_pred), "f1": f1_score(y_test, y_pred, average='macro'), "loss": loss}

# ==============================================================================
# 4. BENCHMARK
# ==============================================================================
def run_benchmark_ultimate(df_raw, target_col):
    print("\n⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive")

    feature_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in feature_cols: feature_cols.remove(target_col)

    X_temp, X_root, y_temp, y_root = train_test_split(df_raw[feature_cols].values, df_raw[target_col].values, test_size=0.05, stratify=df_raw[target_col].values, random_state=RANDOM_STATE)
    X_test, _, y_test, _ = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=RANDOM_STATE)

    scaler = StandardScaler().fit(X_test)
    X_test = scaler.transform(X_test)
    X_root = scaler.transform(X_root)
    X_backdoor = X_test.copy()
    X_backdoor[:, -1] = 99.0
    classes = np.unique(y_test)

    STRATEGIES = ["FedAvg", "Krum", "FLTrust", "Adaptive"]
    if HAS_HDBSCAN: STRATEGIES.insert(3, "FLAME")

    results = []

    for strategy in STRATEGIES:
        print(f"\n🚀 {strategy.upper()} --------------------------------")
        global_model = create_model(RANDOM_STATE)
        global_model.partial_fit(X_test[:100], y_test[:100], classes=classes)
        global_weights = get_weights_mlp(global_model)

        trust_memory = {n: 0.5 for n in NODES}

        for stage_idx, stage_name in enumerate(STAGES_ORDER):
            # --- MEDIÇÃO DE TEMPO (Início) ---
            start_time = time.time()

            stage_data = generate_stage_data(df_raw, feature_cols, target_col, stage_idx, NODES)

            # --- CÁLCULO DA % DE DADOS ATACADOS ---
            # Soma os alphas de cada nó e divide pelo total de nós (média global de veneno)
            total_alpha = sum([d['alpha'] for d in stage_data])
            global_poison_pct = total_alpha / len(NODES)

            server_update = None
            server_norm = 1.0
            if strategy in ["FLTrust", "Adaptive"]:
                server_delta = local_training_mlp(global_weights, X_root, y_root, classes, global_model)
                server_update = server_delta
                server_norm = np.linalg.norm(server_update) + 1e-12

            local_updates = []
            valid_nodes = []
            for node_dict in stage_data:
                X_node = scaler.transform(node_dict['X'])
                delta = local_training_mlp(global_weights, X_node, node_dict['y'], classes, global_model)
                local_updates.append(delta)
                valid_nodes.append(node_dict['node'])

            agg_update = np.zeros_like(global_weights)
            audit_trust = 0.0

            if len(local_updates) > 0:
                if strategy == "FedAvg":
                    agg_update = np.mean(local_updates, axis=0)
                    audit_trust = 1.0

                elif strategy == "Krum":
                    updates_stack = np.stack(local_updates)
                    n = len(local_updates)
                    f = max(1, int(n * 0.25))
                    k = max(1, n - f - 2)
                    dists = cdist(updates_stack, updates_stack, metric='euclidean')
                    scores = []
                    for i in range(n):
                        d_sorted = np.sort(dists[i])
                        scores.append(np.sum(d_sorted[1:k+1]))
                    best_idx = np.argmin(scores)
                    agg_update = updates_stack[best_idx]
                    audit_trust = 1.0 if valid_nodes[best_idx] == "Node1" else 0.0

                elif strategy == "FLTrust":
                    scores = []
                    norm_deltas = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        score = max(0.0, cosine)
                        scores.append(score)
                        norm_deltas.append(delta * (server_norm / norm))
                    if sum(scores) > 0:
                        total = sum(scores)
                        for i, d in enumerate(norm_deltas):
                            agg_update += (scores[i]/total) * d
                        audit_trust = scores[0]

                elif strategy == "FLAME":
                    updates_stack = np.stack(local_updates)
                    dists = cosine_distances(updates_stack)
                    min_size = max(2, int(len(local_updates) * 0.5) + 1)
                    clusterer = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=min_size, allow_single_cluster=True)
                    labels = clusterer.fit_predict(dists)
                    unique, counts = np.unique(labels, return_counts=True)
                    valid_l = unique[unique != -1]
                    if len(valid_l) > 0:
                        best_c = valid_l[np.argmax(counts[unique != -1])]
                        benign_idxs = np.where(labels == best_c)[0]
                    else:
                        benign_idxs = np.arange(len(local_updates))
                    benign_updates = updates_stack[benign_idxs]
                    norms = np.linalg.norm(benign_updates, axis=1)
                    median_norm = np.median(norms)
                    clipped = [u * min(1, median_norm/(np.linalg.norm(u)+1e-12)) for u in benign_updates]
                    if clipped:
                        avg_u = np.mean(clipped, axis=0)
                        noise = np.random.normal(0, FLAME_NOISE_SIGMA * median_norm, avg_u.shape)
                        agg_update = avg_u + noise
                        audit_trust = 1.0 if 0 in benign_idxs else 0.0

                elif strategy == "Adaptive":
                    scores = []
                    valid_ups = []
                    raw_cosines = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        raw_cosines.append(cosine)

                    mu = np.mean(raw_cosines)
                    std = np.std(raw_cosines)
                    dynamic_thresh = max(0.05, mu - (1.0 * std))

                    for i, cosine in enumerate(raw_cosines):
                        node = valid_nodes[i]
                        delta = local_updates[i]
                        norm = np.linalg.norm(delta) + 1e-12
                        if cosine < 0:
                            trust_memory[node] = 0.0
                            new_trust = 0.0
                        else:
                            old = trust_memory[node]
                            if cosine < dynamic_thresh:
                                new_trust = cosine * STRICT_PUNISHMENT
                            else:
                                new_trust = (ADAPTIVE_MOMENTUM * old) + ((1 - ADAPTIVE_MOMENTUM) * cosine)
                            trust_memory[node] = new_trust
                        if new_trust > MIN_TRUST_THRESHOLD:
                            scores.append(new_trust)
                            valid_ups.append(delta * (server_norm / norm))
                        if node == "Node1": audit_trust = new_trust

                    if scores:
                        total = sum(scores)
                        for i, d in enumerate(valid_ups):
                            agg_update += (scores[i]/total) * d

            global_weights += agg_update
            set_weights_mlp(global_model, global_weights, global_model)

            # --- MEDIÇÃO DE TEMPO (Fim) ---
            end_time = time.time()
            elapsed_time = end_time - start_time

            metrics = evaluate(global_model, X_test, y_test)
            pred_bd = global_model.predict(X_backdoor)
            asr = np.mean(pred_bd == 1)

            print(f"{stage_name} | F1: {metrics['f1']:.4f} | ASR: {asr:.4f} | Pois%: {global_poison_pct:.2f} | Time: {elapsed_time:.2f}s")

            results.append({
                "Strategy": strategy,
                "Stage": stage_name,
                "F1_Main": metrics['f1'],
                "ASR_Backdoor": asr,
                "Global_Poisoning_Rate": global_poison_pct, # <--- Nova Coluna
                "Time_Seconds": elapsed_time,               # <--- Nova Coluna
                "Accuracy": metrics['acc'],
                "Trust_N1": audit_trust
            })
            gc.collect()

    return pd.DataFrame(results)

if __name__ == "__main__":
    start_global_time = time.time()

    df_raw, target_col = get_user_data()
    if df_raw is not None:
        if target_col not in df_raw.columns:
            target_col = input("👉 Coluna de Label (ex: attack): ").strip()

        df_res = run_benchmark_ultimate(df_raw, target_col)

        # --- NOMEAÇÃO AUTOMÁTICA COM ARQUIVO DE ENTRADA ---
        # Extrai o nome do arquivo (ex: "Ecobee_Thermostat_mirai_attacks_consolidado")
        input_filename = os.path.basename(DRIVE_FILE_PATH).replace(".csv", "")
        # Monta o nome de saída
        fname = f"{input_filename}_RESULTADOS_ULTIMATE_5_METHODS.csv"

        # Salva localmente e no drive
        dir_path = os.path.dirname(DRIVE_FILE_PATH)

        df_res.to_csv(fname, index=False)
        print(f"\n💾 Salvo Local: {fname}")

        try:
            full_path = os.path.join(dir_path, fname)
            df_res.to_csv(full_path, index=False)
            print(f"✅ Salvo Drive: {full_path}")
        except: pass

    end_global_time = time.time()
    print(f"\n⏱️ Tempo Total de Execução: {(end_global_time - start_global_time)/60:.2f} minutos")

✅ Dataset carregado: (701648, 117)

⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive

🚀 FEDAVG --------------------------------
S0 | F1: 0.3724 | ASR: 0.0000 | Pois%: 0.00 | Time: 38.50s
S1 | F1: 0.3724 | ASR: 0.0000 | Pois%: 0.06 | Time: 39.40s
S2 | F1: 0.3725 | ASR: 0.0002 | Pois%: 0.12 | Time: 38.70s
S3 | F1: 0.3743 | ASR: 0.0003 | Pois%: 0.19 | Time: 38.46s
S4 | F1: 0.3736 | ASR: 0.0011 | Pois%: 0.25 | Time: 38.85s
S5 | F1: 0.3739 | ASR: 0.0001 | Pois%: 0.31 | Time: 39.15s
S6 | F1: 0.3750 | ASR: 0.0032 | Pois%: 0.38 | Time: 38.90s
S7 | F1: 0.3773 | ASR: 0.0002 | Pois%: 0.44 | Time: 38.36s
S8 | F1: 0.3736 | ASR: 0.0000 | Pois%: 0.50 | Time: 39.21s
S9 | F1: 0.3735 | ASR: 0.0019 | Pois%: 0.56 | Time: 39.14s
S10 | F1: 0.3727 | ASR: 0.0008 | Pois%: 0.62 | Time: 39.01s
S11 | F1: 0.3724 | ASR: 0.0049 | Pois%: 0.69 | Time: 39.06s
S12 | F1: 0.3724 | ASR: 0.0057 | Pois%: 0.75 | Time: 38.75s
S13 | F1: 0.3724 | ASR: 0.5867 | Pois%: 0.81 | Time: 38.82s
S14 | F1: 0

In [ ]:
# ---------------------------------------------------------
# 📦 INSTALAÇÃO DE DEPENDÊNCIAS
!pip install hdbscan
# ---------------------------------------------------------

import pandas as pd
import numpy as np
import time
import warnings
import os
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import cdist
from google.colab import drive

# Tenta importar HDBSCAN
try:
    import hdbscan
    HAS_HDBSCAN = True
except ImportError:
    HAS_HDBSCAN = False
    print("⚠️ AVISO: Biblioteca 'hdbscan' não encontrada. FLAME será pulado.")

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURAÇÕES
# ==============================================================================
DRIVE_FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/CSAI4CPS/Datasets/Danmini_Doorbell_gafgyt_attacks_consolidado.csv"

NODES = ["Node1", "Node2", "Node3", "Node4"]
STAGES_ORDER = [f"S{i}" for i in range(17)]
RANDOM_STATE = 42

# Parâmetros Adaptive
ADAPTIVE_MOMENTUM = 0.6
STRICT_PUNISHMENT = 0.1
MIN_TRUST_THRESHOLD = 0.01
FLAME_NOISE_SIGMA = 0.001
HIDDEN_LAYERS = (50, 25)

# ==============================================================================
# 2. UTILITÁRIOS
# ==============================================================================
def get_user_data():
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')
    if not os.path.exists(DRIVE_FILE_PATH):
        print(f"❌ Arquivo não encontrado: {DRIVE_FILE_PATH}")
        return None, None
    try:
        df = pd.read_csv(DRIVE_FILE_PATH)
        print(f"✅ Dataset carregado: {df.shape}")
        return df, "ataque"
    except Exception as e:
        print(f"❌ Erro: {e}")
        return None, None

def generate_stage_data(df_base, feature_cols, target_col, stage_idx, nodes_list):
    alphas_map = {n: 0.0 for n in nodes_list}
    if stage_idx > 0:
        node_idx = (stage_idx - 1) // 4
        step_idx = (stage_idx - 1) % 4
        target_node = nodes_list[min(node_idx, len(nodes_list)-1)]
        current_alpha = [0.25, 0.50, 0.75, 1.00][step_idx]
        for prev_i in range(node_idx):
            alphas_map[nodes_list[prev_i]] = 1.0
        alphas_map[target_node] = current_alpha

    df_shuffled = df_base.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    shards = np.array_split(df_shuffled, len(nodes_list))
    stage_data_list = []

    trigger_feat_idx = -1

    for i, node in enumerate(nodes_list):
        df_node = shards[i].copy()
        y = df_node[target_col].values.astype(int)
        y_poisoned = y.copy()
        X_node = df_node[feature_cols].values.copy()
        alpha = alphas_map[node]

        # Alvo do ataque definido como 1
        if alpha > 0:
            n_poison = int(len(y) * alpha)
            if n_poison > 0:
                poison_idx = np.random.choice(len(y), n_poison, replace=False)
                X_node[poison_idx, trigger_feat_idx] = 99.0
                y_poisoned[poison_idx] = 1

        stage_data_list.append({"node": node, "X": X_node, "y": y_poisoned, "alpha": alpha})
    return stage_data_list

# ==============================================================================
# 3. TREINAMENTO
# ==============================================================================
def create_model(random_state=42):
    return MLPClassifier(hidden_layer_sizes=HIDDEN_LAYERS, activation='relu', solver='adam',
                         alpha=0.0001, batch_size=32, learning_rate_init=0.001,
                         random_state=random_state, max_iter=1)

def get_weights_mlp(model):
    return np.concatenate([w.flatten() for w in model.coefs_] + [b.flatten() for b in model.intercepts_])

def set_weights_mlp(model, flat_weights, ref_model):
    idx = 0
    for i in range(len(model.coefs_)):
        shape = ref_model.coefs_[i].shape
        size = np.prod(shape)
        model.coefs_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size
    for i in range(len(model.intercepts_)):
        shape = ref_model.intercepts_[i].shape
        size = np.prod(shape)
        model.intercepts_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size

def local_training_mlp(global_weights, X, y, classes, ref_model):
    model = create_model(RANDOM_STATE)
    model.partial_fit(X[:5], y[:5], classes=classes)
    set_weights_mlp(model, global_weights.copy(), ref_model)

    for _ in range(5):
        p = np.random.permutation(len(y))
        model.partial_fit(X[p], y[p], classes=classes)
    return get_weights_mlp(model) - global_weights

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    try:
        y_prob = model.predict_proba(X_test)
        loss = log_loss(y_test, y_prob, labels=model.classes_)
    except: loss = 0.0
    return {"acc": accuracy_score(y_test, y_pred), "f1": f1_score(y_test, y_pred, average='macro'), "loss": loss}

# ==============================================================================
# 4. BENCHMARK
# ==============================================================================
def run_benchmark_ultimate(df_raw, target_col):
    print("\n⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive")

    feature_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in feature_cols: feature_cols.remove(target_col)

    X_temp, X_root, y_temp, y_root = train_test_split(df_raw[feature_cols].values, df_raw[target_col].values, test_size=0.05, stratify=df_raw[target_col].values, random_state=RANDOM_STATE)
    X_test, _, y_test, _ = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=RANDOM_STATE)

    scaler = StandardScaler().fit(X_test)
    X_test = scaler.transform(X_test)
    X_root = scaler.transform(X_root)
    X_backdoor = X_test.copy()
    X_backdoor[:, -1] = 99.0
    classes = np.unique(y_test)

    STRATEGIES = ["FedAvg", "Krum", "FLTrust", "Adaptive"]
    if HAS_HDBSCAN: STRATEGIES.insert(3, "FLAME")

    results = []

    for strategy in STRATEGIES:
        print(f"\n🚀 {strategy.upper()} --------------------------------")
        global_model = create_model(RANDOM_STATE)
        global_model.partial_fit(X_test[:100], y_test[:100], classes=classes)
        global_weights = get_weights_mlp(global_model)

        trust_memory = {n: 0.5 for n in NODES}

        for stage_idx, stage_name in enumerate(STAGES_ORDER):
            # --- MEDIÇÃO DE TEMPO (Início) ---
            start_time = time.time()

            stage_data = generate_stage_data(df_raw, feature_cols, target_col, stage_idx, NODES)

            # --- CÁLCULO DA % DE DADOS ATACADOS ---
            # Soma os alphas de cada nó e divide pelo total de nós (média global de veneno)
            total_alpha = sum([d['alpha'] for d in stage_data])
            global_poison_pct = total_alpha / len(NODES)

            server_update = None
            server_norm = 1.0
            if strategy in ["FLTrust", "Adaptive"]:
                server_delta = local_training_mlp(global_weights, X_root, y_root, classes, global_model)
                server_update = server_delta
                server_norm = np.linalg.norm(server_update) + 1e-12

            local_updates = []
            valid_nodes = []
            for node_dict in stage_data:
                X_node = scaler.transform(node_dict['X'])
                delta = local_training_mlp(global_weights, X_node, node_dict['y'], classes, global_model)
                local_updates.append(delta)
                valid_nodes.append(node_dict['node'])

            agg_update = np.zeros_like(global_weights)
            audit_trust = 0.0

            if len(local_updates) > 0:
                if strategy == "FedAvg":
                    agg_update = np.mean(local_updates, axis=0)
                    audit_trust = 1.0

                elif strategy == "Krum":
                    updates_stack = np.stack(local_updates)
                    n = len(local_updates)
                    f = max(1, int(n * 0.25))
                    k = max(1, n - f - 2)
                    dists = cdist(updates_stack, updates_stack, metric='euclidean')
                    scores = []
                    for i in range(n):
                        d_sorted = np.sort(dists[i])
                        scores.append(np.sum(d_sorted[1:k+1]))
                    best_idx = np.argmin(scores)
                    agg_update = updates_stack[best_idx]
                    audit_trust = 1.0 if valid_nodes[best_idx] == "Node1" else 0.0

                elif strategy == "FLTrust":
                    scores = []
                    norm_deltas = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        score = max(0.0, cosine)
                        scores.append(score)
                        norm_deltas.append(delta * (server_norm / norm))
                    if sum(scores) > 0:
                        total = sum(scores)
                        for i, d in enumerate(norm_deltas):
                            agg_update += (scores[i]/total) * d
                        audit_trust = scores[0]

                elif strategy == "FLAME":
                    updates_stack = np.stack(local_updates)
                    dists = cosine_distances(updates_stack)
                    min_size = max(2, int(len(local_updates) * 0.5) + 1)
                    clusterer = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=min_size, allow_single_cluster=True)
                    labels = clusterer.fit_predict(dists)
                    unique, counts = np.unique(labels, return_counts=True)
                    valid_l = unique[unique != -1]
                    if len(valid_l) > 0:
                        best_c = valid_l[np.argmax(counts[unique != -1])]
                        benign_idxs = np.where(labels == best_c)[0]
                    else:
                        benign_idxs = np.arange(len(local_updates))
                    benign_updates = updates_stack[benign_idxs]
                    norms = np.linalg.norm(benign_updates, axis=1)
                    median_norm = np.median(norms)
                    clipped = [u * min(1, median_norm/(np.linalg.norm(u)+1e-12)) for u in benign_updates]
                    if clipped:
                        avg_u = np.mean(clipped, axis=0)
                        noise = np.random.normal(0, FLAME_NOISE_SIGMA * median_norm, avg_u.shape)
                        agg_update = avg_u + noise
                        audit_trust = 1.0 if 0 in benign_idxs else 0.0

                elif strategy == "Adaptive":
                    scores = []
                    valid_ups = []
                    raw_cosines = []
                    for delta in local_updates:
                        norm = np.linalg.norm(delta) + 1e-12
                        cosine = np.dot(server_update, delta) / (server_norm * norm)
                        raw_cosines.append(cosine)

                    mu = np.mean(raw_cosines)
                    std = np.std(raw_cosines)
                    dynamic_thresh = max(0.05, mu - (1.0 * std))

                    for i, cosine in enumerate(raw_cosines):
                        node = valid_nodes[i]
                        delta = local_updates[i]
                        norm = np.linalg.norm(delta) + 1e-12
                        if cosine < 0:
                            trust_memory[node] = 0.0
                            new_trust = 0.0
                        else:
                            old = trust_memory[node]
                            if cosine < dynamic_thresh:
                                new_trust = cosine * STRICT_PUNISHMENT
                            else:
                                new_trust = (ADAPTIVE_MOMENTUM * old) + ((1 - ADAPTIVE_MOMENTUM) * cosine)
                            trust_memory[node] = new_trust
                        if new_trust > MIN_TRUST_THRESHOLD:
                            scores.append(new_trust)
                            valid_ups.append(delta * (server_norm / norm))
                        if node == "Node1": audit_trust = new_trust

                    if scores:
                        total = sum(scores)
                        for i, d in enumerate(valid_ups):
                            agg_update += (scores[i]/total) * d

            global_weights += agg_update
            set_weights_mlp(global_model, global_weights, global_model)

            # --- MEDIÇÃO DE TEMPO (Fim) ---
            end_time = time.time()
            elapsed_time = end_time - start_time

            metrics = evaluate(global_model, X_test, y_test)
            pred_bd = global_model.predict(X_backdoor)
            asr = np.mean(pred_bd == 1)

            print(f"{stage_name} | F1: {metrics['f1']:.4f} | ASR: {asr:.4f} | Pois%: {global_poison_pct:.2f} | Time: {elapsed_time:.2f}s")

            results.append({
                "Strategy": strategy,
                "Stage": stage_name,
                "F1_Main": metrics['f1'],
                "ASR_Backdoor": asr,
                "Global_Poisoning_Rate": global_poison_pct, # <--- Nova Coluna
                "Time_Seconds": elapsed_time,               # <--- Nova Coluna
                "Accuracy": metrics['acc'],
                "Trust_N1": audit_trust
            })
            gc.collect()

    return pd.DataFrame(results)

if __name__ == "__main__":
    start_global_time = time.time()

    df_raw, target_col = get_user_data()
    if df_raw is not None:
        if target_col not in df_raw.columns:
            target_col = input("👉 Coluna de Label (ex: attack): ").strip()

        df_res = run_benchmark_ultimate(df_raw, target_col)

        # --- NOMEAÇÃO AUTOMÁTICA COM ARQUIVO DE ENTRADA ---
        # Extrai o nome do arquivo (ex: "Ecobee_Thermostat_mirai_attacks_consolidado")
        input_filename = os.path.basename(DRIVE_FILE_PATH).replace(".csv", "")
        # Monta o nome de saída
        fname = f"{input_filename}_RESULTADOS_ULTIMATE_5_METHODS.csv"

        # Salva localmente e no drive
        dir_path = os.path.dirname(DRIVE_FILE_PATH)

        df_res.to_csv(fname, index=False)
        print(f"\n💾 Salvo Local: {fname}")

        try:
            full_path = os.path.join(dir_path, fname)
            df_res.to_csv(full_path, index=False)
            print(f"✅ Salvo Drive: {full_path}")
        except: pass

    end_global_time = time.time()
    print(f"\n⏱️ Tempo Total de Execução: {(end_global_time - start_global_time)/60:.2f} minutos")

✅ Dataset carregado: (732396, 117)

⚔️ INICIANDO ULTIMATE BENCHMARK: FedAvg vs Krum vs FLTrust vs FLAME vs Adaptive

🚀 FEDAVG --------------------------------
S0 | F1: 0.4395 | ASR: 0.0000 | Pois%: 0.00 | Time: 40.01s
S1 | F1: 0.4395 | ASR: 0.0000 | Pois%: 0.06 | Time: 40.28s
S2 | F1: 0.4395 | ASR: 0.0000 | Pois%: 0.12 | Time: 40.84s
S3 | F1: 0.4394 | ASR: 0.0000 | Pois%: 0.19 | Time: 40.12s
S4 | F1: 0.5232 | ASR: 0.0095 | Pois%: 0.25 | Time: 40.49s
S5 | F1: 0.4464 | ASR: 0.0002 | Pois%: 0.31 | Time: 40.61s
S6 | F1: 0.5133 | ASR: 0.0015 | Pois%: 0.38 | Time: 41.02s
S7 | F1: 0.1779 | ASR: 0.0065 | Pois%: 0.44 | Time: 40.79s
S8 | F1: 0.1778 | ASR: 0.3792 | Pois%: 0.50 | Time: 40.74s
S9 | F1: 0.1778 | ASR: 0.9999 | Pois%: 0.56 | Time: 40.10s
S10 | F1: 0.1778 | ASR: 1.0000 | Pois%: 0.62 | Time: 40.83s
S11 | F1: 0.1777 | ASR: 1.0000 | Pois%: 0.69 | Time: 41.32s
S12 | F1: 0.1777 | ASR: 1.0000 | Pois%: 0.75 | Time: 40.94s
S13 | F1: 0.1777 | ASR: 1.0000 | Pois%: 0.81 | Time: 40.34s
S14 | F1: 0